In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e11/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e11/train.csv
/kaggle/input/competitions/playground-series-s6e11/test.csv
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/__huggingface_repos__.json
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/__notebook__.ipynb
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/__output__.json
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/__results__.html
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/custom.css
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp3.npz
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp4.npz
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp5.npz
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp7_glmboost.npz
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_opt2.npz
/kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/submission.csv
/kaggle/input/datasets/

In [2]:
# Step 1: load the tables and look at what is inside
SKIP_SUPERSEDED = True  # Skip 0-weight superseded early models to save >90 mins of GPU quota

if "train" not in globals():
    d = "/kaggle/input/competitions/playground-series-s6e11/"
    train = pd.read_csv(d + "train.csv")
    test = pd.read_csv(d + "test.csv")
    print("rows, columns:", train.shape, test.shape)
    print("share of fraud rows:", train["is_fraud"].mean())

import shutil, glob
for _npz in ["preds_backup.npz", "preds_opt2.npz", "preds_exp3.npz", "preds_exp4.npz", "preds_exp5.npz", "preds_exp6_meta.npz", "preds_exp7_glmboost.npz", "preds_exp9.npz"]:
    _p = "/kaggle/working/" + _npz
    if not os.path.exists(_p):
        _hits = sorted(glob.glob(f"/kaggle/input/**/{_npz}", recursive=True))
        if _hits:
            shutil.copy2(_hits[0], _p)
            print(f"Copied {_npz} from {_hits[0]} to /kaggle/working/")
    if os.path.exists(_p):
        _b = np.load(_p)
        for _k in _b.files:
            if _k not in globals():
                globals()[_k] = _b[_k].astype(np.float64)
        print("Restored cache:", _npz, len(_b.files))


rows, columns: (699960, 26) (299984, 25)
share of fraud rows: 0.016699525687182123
Copied preds_opt2.npz from /kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_opt2.npz to /kaggle/working/
Restored cache: preds_opt2.npz 4
Copied preds_exp3.npz from /kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp3.npz to /kaggle/working/
Restored cache: preds_exp3.npz 8
Copied preds_exp4.npz from /kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp4.npz to /kaggle/working/
Restored cache: preds_exp4.npz 4
Copied preds_exp5.npz from /kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp5.npz to /kaggle/working/
Restored cache: preds_exp5.npz 6
Copied preds_exp7_glmboost.npz from /kaggle/input/notebooks/phankhnhng/notebook2d1e76465a/preds_exp7_glmboost.npz to /kaggle/working/
Restored cache: preds_exp7_glmboost.npz 40
Copied preds_exp9.npz from /kaggle/input/datasets/phankhnhng/s6e11-exp9-cache/preds_exp9.npz to /kaggle/working/
Restored cache: preds_exp9.npz 

In [3]:
# Step 2: Feature Engineering + 5-fold LightGBM
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def make_features(df):
    out = df.copy()
    out["amount_to_balance"] = out["amount_usd"] / (out["account_balance_usd"] + 1.0)
    out["balance_after_txn"] = out["account_balance_usd"] - out["amount_usd"]
    out["txn_burst_rate"] = out["txn_count_last_24h"] / (out["hours_since_last_txn"] + 0.1)
    flag_cols = [
        "is_foreign_transaction", "is_new_merchant", "used_vpn",
        "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
    ]
    out["risk_flag_count"] = out[flag_cols].sum(axis=1)
    out["amount_x_merchant_risk"] = out["amount_usd"] * out["merchant_risk_score"]
    return out

train_fe = make_features(train)
test_fe = make_features(test)

text_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for c in text_cols:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = pd.Categorical(test_fe[c], categories=train_fe[c].cat.categories)

X = train_fe.drop(columns=["id", "is_fraud"])
y = train_fe["is_fraud"]
X_test = test_fe.drop(columns=["id"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

if "oof_preds" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=fold, verbose=-1, n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / 5

if "oof_preds" not in globals():
    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
lgb_oof = oof_preds.copy()
lgb_test = test_preds.copy()

In [4]:
# Step 3: 5-fold XGBoost
import xgboost as xgb

if "xgb_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    xgb_oof = np.zeros(len(train_fe))
    xgb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        xgb_model = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=100,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            device="cuda",
            enable_categorical=True,
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=fold,
            n_jobs=-1,
        )
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        xgb_oof[va_idx] = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test += xgb_model.predict_proba(X_test)[:, 1] / 5

if "xgb_oof" not in globals():
    xgb_oof = np.zeros(len(train_fe))
    xgb_test = np.zeros(len(test_fe))

In [5]:
# Step 4: 5-fold CatBoost + 3-Model Blend (LightGBM + XGBoost + CatBoost)
from catboost import CatBoostClassifier

if "cb_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    X_cb = X.copy()
    X_test_cb = X_test.copy()
    for c in text_cols:
        X_cb[c] = X_cb[c].cat.codes
        X_test_cb[c] = X_test_cb[c].cat.codes

    cb_oof = np.zeros(len(train_fe))
    cb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y)):
        X_tr, y_tr = X_cb.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_cb.iloc[va_idx], y.iloc[va_idx]

        cb_model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.06,
            depth=6,
            cat_features=text_cols,
            one_hot_max_size=25,
            eval_metric="AUC",
            random_seed=fold,
            task_type="GPU",
            verbose=False,
        )
        cb_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=50,
            verbose=False,
        )

        cb_oof[va_idx] = cb_model.predict_proba(X_va)[:, 1]
        cb_test += cb_model.predict_proba(X_test_cb)[:, 1] / 5

if "cb_oof" not in globals():
    cb_oof = np.zeros(len(train_fe))
    cb_test = np.zeros(len(test_fe))
blend3_oof = np.zeros(len(train_fe))
blend3_test = np.zeros(len(test_fe))

In [6]:
# Step 5: Test 12 Non-Leaky Group Aggregation Z-Scores on 5-fold LightGBM
if "lgb_grp_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    def add_group_aggregations(tr_df, te_df):
        full = pd.concat([tr_df.drop(columns=["is_fraud"]), te_df], ignore_index=True)
        for grp in ["merchant_category", "channel", "auth_method"]:
            for num in ["amount_usd", "velocity_score", "merchant_risk_score", "distance_from_home_km"]:
                means = full.groupby(grp, observed=False)[num].transform("mean")
                stds = full.groupby(grp, observed=False)[num].transform("std") + 1e-5
                full[f"{num}_z_{grp}"] = ((full[num] - means) / stds).astype(np.float32)
        tr_out = full.iloc[:len(tr_df)].drop(columns=["id"]).copy()
        te_out = full.iloc[len(tr_df):].drop(columns=["id"]).copy().reset_index(drop=True)
        return tr_out, te_out
    
    X_grp, X_test_grp = add_group_aggregations(train_fe, test_fe)
    for c in text_cols:
        X_grp[c] = X_grp[c].astype("category")
        X_test_grp[c] = pd.Categorical(X_test_grp[c], categories=X_grp[c].cat.categories)
    
    lgb_grp_oof = np.zeros(len(train_fe))
    lgb_grp_test = np.zeros(len(test_fe))
    grp_imp = np.zeros(X_grp.shape[1])
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_grp, y)):
        X_tr, y_tr = X_grp.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_grp.iloc[va_idx], y.iloc[va_idx]
    
        model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=fold, verbose=-1, n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        lgb_grp_oof[va_idx] = model.predict_proba(X_va)[:, 1]
        lgb_grp_test += model.predict_proba(X_test_grp)[:, 1] / 5
        grp_imp += model.feature_importances_ / 5
        print(f"LGB+GroupAgg Fold {fold} AUC: {roc_auc_score(y_va, lgb_grp_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    grp_auc = roc_auc_score(y, lgb_grp_oof)
    print(f"\nLightGBM baseline OOF AUC:    {roc_auc_score(y, lgb_oof):.5f}")
    print(f"LightGBM + GroupAgg OOF AUC:  {grp_auc:.5f} (diff: {grp_auc - roc_auc_score(y, lgb_oof):+.5f})")
    
    best_w, best_auc = 0.0, roc_auc_score(y, blend3_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
        cand = w * lgb_grp_oof + (1 - w) * blend3_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} LGB_Grp + {1-w:.1f} Prev_3Way): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    final_test = best_w * lgb_grp_test + (1 - best_w) * blend3_test
    print("\nTop 12 features:")
    print(pd.Series(grp_imp, index=X_grp.columns).sort_values(ascending=False).head(12).round(1))
if "lgb_grp_oof" not in globals():
    lgb_grp_oof = np.zeros(len(train_fe))
    lgb_grp_test = np.zeros(len(test_fe))


In [7]:
# Step 6: Digit-Level Features + Shallow Trees (num_leaves=4, max_depth=2)
def add_digit_features(df):
    out = df.copy()
    float_cols = [
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "velocity_score", "merchant_risk_score"
    ]
    for col in float_cols:
        v = out[col].fillna(0).abs()
        int_v = np.floor(v).astype(np.int64)
        cents = np.round((v - int_v) * 100).astype(np.int32)
        out[f"{col}_cents"] = cents
        out[f"{col}_tenth"] = (cents // 10).astype(np.int32)
        out[f"{col}_hundredth"] = (cents % 10).astype(np.int32)
        out[f"{col}_ones"] = (int_v % 10).astype(np.int32)
        out[f"{col}_tens"] = ((int_v // 10) % 10).astype(np.int32)
        out[f"{col}_log1p"] = np.log1p(v).astype(np.float32)
    return out

X_dig = add_digit_features(train_fe.drop(columns=["id", "is_fraud"]))
X_test_dig = add_digit_features(test_fe.drop(columns=["id"]))

for c in text_cols:
    X_dig[c] = X_dig[c].astype("category")
    X_test_dig[c] = pd.Categorical(X_test_dig[c], categories=X_dig[c].cat.categories)

if "lgb_shallow_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    lgb_shallow_oof = np.zeros(len(train_fe))
    lgb_shallow_test = np.zeros(len(test_fe))
    sh_imp = np.zeros(X_dig.shape[1])
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr, y_tr = X_dig.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_dig.iloc[va_idx], y.iloc[va_idx]
    
        model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.05,
            num_leaves=4,
            max_depth=2,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=fold,
            verbose=-1,
            n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        lgb_shallow_oof[va_idx] = model.predict_proba(X_va)[:, 1]
        lgb_shallow_test += model.predict_proba(X_test_dig)[:, 1] / 5
        sh_imp += model.feature_importances_ / 5
        print(f"Shallow+Digits Fold {fold} AUC: {roc_auc_score(y_va, lgb_shallow_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    sh_auc = roc_auc_score(y, lgb_shallow_oof)
    print(f"\nPrevious Deep LightGBM (31 leaves):  {roc_auc_score(y, lgb_oof):.5f}")
    print(f"Previous 4-Model Ensemble:           {roc_auc_score(y, 0.2 * lgb_grp_oof + 0.8 * blend3_oof):.5f}")
    print(f"NEW Shallow LGB (4 leaves) + Digits: {sh_auc:.5f} (vs deep LGB: {sh_auc - roc_auc_score(y, lgb_oof):+.5f})")
    
    prev_best_oof = 0.2 * lgb_grp_oof + 0.8 * blend3_oof
    prev_best_test = 0.2 * lgb_grp_test + 0.8 * blend3_test
    
    best_w, best_auc = 0.0, roc_auc_score(y, prev_best_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        cand = w * lgb_shallow_oof + (1 - w) * prev_best_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} Shallow_Digits + {1-w:.1f} Prev_Best): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    print(f"\nNew Best Ensemble OOF AUC: {best_auc:.5f}")
    final_test = best_w * lgb_shallow_test + (1 - best_w) * prev_best_test
    print("\nTop 12 features in Shallow+Digits:")
    print(pd.Series(sh_imp, index=X_dig.columns).sort_values(ascending=False).head(12).round(1))
if "lgb_shallow_oof" not in globals():
    lgb_shallow_oof = np.zeros(len(train_fe))
    lgb_shallow_test = np.zeros(len(test_fe))
    print("[Skip] lgb_shallow_oof skipped (cached predictions loaded)")


[Skip] lgb_shallow_oof skipped (cached predictions loaded)


In [8]:
# Step 7: Nested Out-of-Fold Target Encoding on Digit & Category Keys + Shallow Trees
te_keys = [
    "amount_usd_cents", "hours_since_last_txn_cents", "distance_from_home_km_cents",
    "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents",
    "amount_usd_ones", "amount_usd_tens", "hours_since_last_txn_ones",
    "merchant_category", "card_type", "auth_method", "channel", "device_type"
]

if "lgb_nte_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    lgb_nte_oof = np.zeros(len(train_fe))
    lgb_nte_test = np.zeros(len(test_fe))
    inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    smooth = 20.0
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr = X_dig.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx]
        X_va = X_dig.iloc[va_idx].copy()
        y_va = y.iloc[va_idx]
        X_te = X_test_dig.copy()
    
        global_prior = y_tr.mean()
        tr_te = pd.DataFrame(index=X_tr.index)
        va_te = pd.DataFrame(index=X_va.index)
        te_te = pd.DataFrame(index=X_te.index)
    
        for col in te_keys:
            col_name = f"nte_{col}"
            tr_vals = np.full(len(X_tr), global_prior, dtype=np.float32)
    
            # Nested inner folds: each row is encoded WITHOUT its own label!
            s_tr = X_tr[col].astype(str)
            for in_tr, in_va in inner_skf.split(X_tr, y_tr):
                stats = y_tr.iloc[in_tr].groupby(s_tr.iloc[in_tr]).agg(["count", "sum"])
                te_map = (stats["sum"] + smooth * global_prior) / (stats["count"] + smooth)
                tr_vals[in_va] = s_tr.iloc[in_va].map(te_map).fillna(global_prior).values
    
            tr_te[col_name] = tr_vals
    
            # Outer val & test use all of (X_tr, y_tr)
            stats_full = y_tr.groupby(s_tr).agg(["count", "sum"])
            te_map_full = (stats_full["sum"] + smooth * global_prior) / (stats_full["count"] + smooth)
            va_te[col_name] = X_va[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
            te_te[col_name] = X_te[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
    
        X_tr_full = pd.concat([X_tr, tr_te], axis=1)
        X_va_full = pd.concat([X_va, va_te], axis=1)
        X_te_full = pd.concat([X_te, te_te], axis=1)
    
        model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.04,
            num_leaves=3,
            max_depth=2,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=fold,
            verbose=-1,
            n_jobs=-1,
        )
        model.fit(
            X_tr_full, y_tr,
            eval_set=[(X_va_full, y_va)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        lgb_nte_oof[va_idx] = model.predict_proba(X_va_full)[:, 1]
        lgb_nte_test += model.predict_proba(X_te_full)[:, 1] / 5
        print(f"Shallow+Digits+NestedTE Fold {fold} AUC: {roc_auc_score(y_va, lgb_nte_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    nte_auc = roc_auc_score(y, lgb_nte_oof)
    print(f"\nShallow LGB + Digits alone:           {roc_auc_score(y, lgb_shallow_oof):.5f}")
    print(f"NEW Shallow LGB + Digits + Nested TE: {nte_auc:.5f} (diff: {nte_auc - roc_auc_score(y, lgb_shallow_oof):+.5f})")
    
    prev_best_oof = 0.8 * lgb_shallow_oof + 0.2 * (0.2 * lgb_grp_oof + 0.8 * blend3_oof)
    prev_best_test = 0.8 * lgb_shallow_test + 0.2 * (0.2 * lgb_grp_test + 0.8 * blend3_test)
    
    best_w, best_auc = 0.0, roc_auc_score(y, prev_best_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
        cand = w * lgb_nte_oof + (1 - w) * prev_best_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} NestedTE + {1-w:.1f} Prev_Best): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    print(f"\nNew Best Ensemble OOF AUC: {best_auc:.5f}")
    final_test = best_w * lgb_nte_test + (1 - best_w) * prev_best_test
if "lgb_nte_oof" not in globals():
    lgb_nte_oof = np.zeros(len(train_fe))
    lgb_nte_test = np.zeros(len(test_fe))


In [9]:
# Step 8: Shallow XGBoost (max_depth=2) & Shallow CatBoost (depth=3) on Digits + Nested TE
from catboost import CatBoostClassifier
import xgboost as xgb

te_keys = [
    "amount_usd_cents", "hours_since_last_txn_cents", "distance_from_home_km_cents",
    "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents",
    "amount_usd_ones", "amount_usd_tens", "hours_since_last_txn_ones",
    "merchant_category", "card_type", "auth_method", "channel", "device_type"
]
inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smooth = 20.0

def get_fold_nte(tr_idx, va_idx):
    X_tr = X_dig.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]
    X_va = X_dig.iloc[va_idx].copy()
    X_te = X_test_dig.copy()
    global_prior = y_tr.mean()
    tr_te = pd.DataFrame(index=X_tr.index)
    va_te = pd.DataFrame(index=X_va.index)
    te_te = pd.DataFrame(index=X_te.index)
    for col in te_keys:
        col_name = f"nte_{col}"
        tr_vals = np.full(len(X_tr), global_prior, dtype=np.float32)
        s_tr = X_tr[col].astype(str)
        for in_tr, in_va in inner_skf.split(X_tr, y_tr):
            stats = y_tr.iloc[in_tr].groupby(s_tr.iloc[in_tr]).agg(["count", "sum"])
            te_map = (stats["sum"] + smooth * global_prior) / (stats["count"] + smooth)
            tr_vals[in_va] = s_tr.iloc[in_va].map(te_map).fillna(global_prior).values
        tr_te[col_name] = tr_vals
        stats_full = y_tr.groupby(s_tr).agg(["count", "sum"])
        te_map_full = (stats_full["sum"] + smooth * global_prior) / (stats_full["count"] + smooth)
        va_te[col_name] = X_va[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
        te_te[col_name] = X_te[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
    return (
        pd.concat([X_tr, tr_te], axis=1),
        pd.concat([X_va, va_te], axis=1),
        pd.concat([X_te, te_te], axis=1),
    )

if "xgb_sh_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    xgb_sh_oof = np.zeros(len(train_fe))
    xgb_sh_test = np.zeros(len(test_fe))
    cb_sh_oof = np.zeros(len(train_fe))
    cb_sh_test = np.zeros(len(test_fe))
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr_f, X_va_f, X_te_f = get_fold_nte(tr_idx, va_idx)
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
        # 1. Shallow XGBoost (max_depth=2)
        xm = xgb.XGBClassifier(
            n_estimators=4000, learning_rate=0.05, max_depth=2,
            min_child_weight=100, subsample=0.8, colsample_bytree=0.8,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=100,
            random_state=fold, n_jobs=-1,
        )
        xm.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], verbose=False)
        xgb_sh_oof[va_idx] = xm.predict_proba(X_va_f)[:, 1]
        xgb_sh_test += xm.predict_proba(X_te_f)[:, 1] / 5
    
        # 2. Shallow CatBoost (depth=3)
        X_tr_cb, X_va_cb, X_te_cb = X_tr_f.copy(), X_va_f.copy(), X_te_f.copy()
        for c in text_cols:
            X_tr_cb[c] = X_tr_cb[c].cat.codes
            X_va_cb[c] = X_va_cb[c].cat.codes
            X_te_cb[c] = X_te_cb[c].cat.codes
    
        cm = CatBoostClassifier(
            iterations=3000, learning_rate=0.06, depth=3,
            cat_features=text_cols, one_hot_max_size=25,
            eval_metric="AUC", random_seed=fold, task_type="GPU", verbose=False,
        )
        cm.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), early_stopping_rounds=100, verbose=False)
        cb_sh_oof[va_idx] = cm.predict_proba(X_va_cb)[:, 1]
        cb_sh_test += cm.predict_proba(X_te_cb)[:, 1] / 5
    
        print(f"Fold {fold} | Shallow XGB: {roc_auc_score(y_va, xgb_sh_oof[va_idx]):.5f} (tree {xm.best_iteration}) | Shallow CB: {roc_auc_score(y_va, cb_sh_oof[va_idx]):.5f} (tree {cm.get_best_iteration()})")
    
    print("\n--- Shallow Models on Digits + Nested TE (5-fold OOF AUC) ---")
    print(f"Shallow LightGBM (3 leaves): {roc_auc_score(y, lgb_nte_oof):.5f}")
    print(f"Shallow XGBoost (depth=2):   {roc_auc_score(y, xgb_sh_oof):.5f}")
    print(f"Shallow CatBoost (depth=3):  {roc_auc_score(y, cb_sh_oof):.5f}")
    
    # Optimize 5-member ensemble (Shallow LGB+NTE, Shallow XGB+NTE, Shallow CB+NTE, Shallow LGB+Digits, Deep 3-Way)
    deep_oof = 0.2 * lgb_grp_oof + 0.8 * blend3_oof
    deep_test = 0.2 * lgb_grp_test + 0.8 * blend3_test
    
    from scipy.optimize import minimize
    models_oof = np.column_stack([lgb_nte_oof, xgb_sh_oof, cb_sh_oof, lgb_shallow_oof, deep_oof])
    models_test = np.column_stack([lgb_nte_test, xgb_sh_test, cb_sh_test, lgb_shallow_test, deep_test])
    
    def loss(weights):
        w = np.maximum(weights, 0)
        w = w / (w.sum() + 1e-12)
        return -roc_auc_score(y, models_oof @ w)
    
    res = minimize(loss, [0.3, 0.2, 0.2, 0.2, 0.1], method="Nelder-Mead")
    best_w = np.maximum(res.x, 0)
    best_w /= best_w.sum()
    names = ["Shallow_LGB_NTE", "Shallow_XGB_NTE", "Shallow_CB_NTE", "Shallow_LGB_Dig", "Deep_Blend"]
    for n, w in zip(names, best_w):
        print(f"  Weight {n:16s}: {w:.3f}")
    
    final_oof = models_oof @ best_w
    final_test = models_test @ best_w
    print(f"\nNEW 5-Member Ensemble OOF AUC: {roc_auc_score(y, final_oof):.5f}")
if "xgb_sh_oof" not in globals():
    xgb_sh_oof = np.zeros(len(train_fe))
    xgb_sh_test = np.zeros(len(test_fe))
    cb_sh_oof = np.zeros(len(train_fe))
    cb_sh_test = np.zeros(len(test_fe))
    deep_oof = np.zeros(len(train_fe))
    deep_test = np.zeros(len(test_fe))


In [10]:
# Step 9: Finer Digit Keys + Frequency + Fast Nested TE + 2-Leaf Stumps & 3-Leaf Trees
def add_finer_digits(df):
    out = df.copy()
    float_cols = [
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "velocity_score", "merchant_risk_score"
    ]
    for col in float_cols:
        v = out[col].fillna(0).abs()
        int_v = np.floor(v).astype(np.int64)
        cents = np.round((v - int_v) * 100).astype(np.int32)
        out[f"{col}_int"] = int_v.astype(np.int32)
        out[f"{col}_div10"] = (int_v // 10).astype(np.int32)
        out[f"{col}_div100"] = (int_v // 100).astype(np.int32)
        out[f"{col}_hundreds"] = ((int_v // 100) % 10).astype(np.int32)
        out[f"{col}_thousands"] = ((int_v // 1000) % 10).astype(np.int32)
        out[f"{col}_is_round_cents"] = (cents == 0).astype(np.int8)
        out[f"{col}_is_99_cents"] = (cents == 99).astype(np.int8)
        # Leading digit (1..9, or 0 for 0.0) using fast vectorized log10
        v_pos = np.where(v > 0, v, 1.0)
        lead = np.floor(np.where(v > 0, v / (10.0 ** np.floor(np.log10(v_pos))), 0.0) + 1e-9)
        out[f"{col}_first_digit"] = np.clip(lead, 0, 9).astype(np.int8)
    for col in ["customer_age", "card_age_months"]:
        out[f"{col}_ones"] = (out[col] % 10).astype(np.int32)
        out[f"{col}_tens"] = (out[col] // 10).astype(np.int32)
    return out

if "X_dig2" not in globals():
    X_dig2 = add_finer_digits(X_dig)
    X_test_dig2 = add_finer_digits(X_test_dig)

    # Frequency encoding on key columns (train+test value counts, zero label leakage)
    n_tr = len(X_dig2)
    freq_cols = [
        "amount_usd_cents", "hours_since_last_txn_cents", "account_balance_usd_cents",
        "amount_usd_int", "hours_since_last_txn_int", "distance_from_home_km_int",
        "customer_age", "card_age_months", "merchant_risk_score_int", "velocity_score_int"
    ]
    for col in freq_cols:
        comb = pd.concat([X_dig2[col], X_test_dig2[col]], ignore_index=True)
        vc = comb.value_counts()
        X_dig2[f"{col}_freq"] = X_dig2[col].map(vc).astype(np.float32)
        X_test_dig2[f"{col}_freq"] = X_test_dig2[col].map(vc).astype(np.float32)

    # Pre-factorize all discrete & digit keys for ultra-fast np.bincount Nested TE
nte2_keys = [
    "merchant_category", "card_type", "auth_method", "channel", "device_type",
    "time_of_day_hour", "day_of_week", "cvv_retry_count", "prior_disputes",
    "customer_age", "card_age_months", "txn_count_last_24h",
    "amount_usd_cents", "amount_usd_tenth", "amount_usd_hundredth", "amount_usd_ones", "amount_usd_tens", "amount_usd_int", "amount_usd_div10", "amount_usd_div100", "amount_usd_first_digit",
    "hours_since_last_txn_cents", "hours_since_last_txn_ones", "hours_since_last_txn_tens", "hours_since_last_txn_int", "hours_since_last_txn_first_digit",
    "distance_from_home_km_cents", "distance_from_home_km_ones", "distance_from_home_km_tens", "distance_from_home_km_div10", "distance_from_home_km_first_digit",
    "account_balance_usd_cents", "account_balance_usd_ones", "account_balance_usd_tens", "account_balance_usd_div10", "account_balance_usd_div100", "account_balance_usd_first_digit",
    "velocity_score_cents", "velocity_score_ones", "velocity_score_tens", "velocity_score_int", "velocity_score_first_digit",
    "merchant_risk_score_cents", "merchant_risk_score_ones", "merchant_risk_score_tens", "merchant_risk_score_int", "merchant_risk_score_first_digit",
]

n_tr = len(X_dig2)
if "codes_tr_dict" not in globals():
    codes_tr_dict, codes_te_dict, ncats_dict = {}, {}, {}
    for col in nte2_keys:
        comb = pd.concat([X_dig2[col].astype(str), X_test_dig2[col].astype(str)], ignore_index=True)
        codes, _ = pd.factorize(comb, sort=False)
        codes_tr_dict[col] = codes[:n_tr]
        codes_te_dict[col] = codes[n_tr:]
        ncats_dict[col] = int(codes.max()) + 1

inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smooth = 20.0
y_arr = y.values.astype(np.float64)

if "lgb_dig2_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    lgb_stump_oof = np.zeros(n_tr)
    lgb_stump_test = np.zeros(len(X_test_dig2))
    lgb_dig2_oof = np.zeros(n_tr)
    lgb_dig2_test = np.zeros(len(X_test_dig2))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        global_prior = y_tr_arr.mean()

        tr_te, va_te, te_te = {}, {}, {}
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        for col in nte2_keys:
            c_all_tr = codes_tr_dict[col]
            c_tr = c_all_tr[tr_idx]
            c_va = c_all_tr[va_idx]
            c_te = codes_te_dict[col]
            nc = ncats_dict[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth * global_prior) / (cnt + smooth)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth * global_prior) / (cnt_f + smooth)

            col_name = f"nte2_{col}"
            tr_te[col_name] = tr_vals
            va_te[col_name] = te_map_f[c_va].astype(np.float32)
            te_te[col_name] = te_map_f[c_te].astype(np.float32)

        X_tr_f = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te, index=X_dig2.index[tr_idx])], axis=1)
        X_va_f = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te, index=X_dig2.index[va_idx])], axis=1)
        X_te_f = pd.concat([X_test_dig2, pd.DataFrame(te_te, index=X_test_dig2.index)], axis=1)

        # Model A: 2-leaf decision stumps (pure additive GAM, zero interaction overfitting)
        m_stump = lgb.LGBMClassifier(
            n_estimators=8000, learning_rate=0.05, num_leaves=2, max_depth=1,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=fold, verbose=-1, n_jobs=-1,
        )
        m_stump.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_stump_oof[va_idx] = m_stump.predict_proba(X_va_f)[:, 1]
        lgb_stump_test += m_stump.predict_proba(X_te_f)[:, 1] / 5

        # Model B: 3-leaf shallow trees on upgraded features
        m_dig2 = lgb.LGBMClassifier(
            n_estimators=6000, learning_rate=0.04, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
            random_state=fold + 100, verbose=-1, n_jobs=-1,
        )
        m_dig2.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_dig2_oof[va_idx] = m_dig2.predict_proba(X_va_f)[:, 1]
        lgb_dig2_test += m_dig2.predict_proba(X_te_f)[:, 1] / 5

        print(f"Fold {fold} | 2-Leaf Stump: {roc_auc_score(y_va, lgb_stump_oof[va_idx]):.5f} (tree {m_stump.best_iteration_}) | 3-Leaf Dig2: {roc_auc_score(y_va, lgb_dig2_oof[va_idx]):.5f} (tree {m_dig2.best_iteration_})")

if "lgb_dig2_oof" not in globals():
    lgb_stump_oof = np.zeros(len(X_dig2))
    lgb_stump_test = np.zeros(len(X_test_dig2))
    lgb_dig2_oof = np.zeros(len(X_dig2))
    lgb_dig2_test = np.zeros(len(X_test_dig2))
print("X_dig2 ready:", X_dig2.shape, "| nte2_keys:", len(nte2_keys))

X_dig2 ready: (699960, 127) | nte2_keys: 47


In [11]:
# Step 10: Option 2 — Spline GLM + PyTorch GPU Residual MLP on X_dig2 + 47-Key Nested TE
import gc
import time
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def signed_log1p(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def safe_logit(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "glm_oof" not in globals() or "mlp_oof" not in globals():
    t0_all = time.time()
    n_tr = len(X_dig2)
    n_te = len(X_test_dig2)
    glm_oof = np.zeros(n_tr, dtype=np.float64)
    glm_test = np.zeros(n_te, dtype=np.float64)
    mlp_oof = np.zeros(n_tr, dtype=np.float64)
    mlp_test = np.zeros(n_te, dtype=np.float64)

    num_cols = [c for c in X_dig2.columns if c not in text_cols]
    log_skew_cols = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew_cols.add(c)

    # Pre-build signed-log1p numeric base matrices once (float32)
    base_tr = X_dig2[num_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base_te = X_test_dig2[num_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num_cols):
        if c in log_skew_cols:
            base_tr[:, j] = signed_log1p(base_tr[:, j])
            base_te[:, j] = signed_log1p(base_te[:, j])
    base_tr = np.nan_to_num(base_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base_te = np.nan_to_num(base_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline_base_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline_base_idx = [num_cols.index(c) for c in spline_base_cols]

    oh_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh_enc.fit_transform(X_dig2[oh_cols].astype(str))
    _oh_te = oh_enc.transform(X_test_dig2[oh_cols].astype(str))
    oh_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    emb_cols = [
        "merchant_category", "card_type", "auth_method", "channel", "device_type",
        "time_of_day_hour", "day_of_week", "cvv_retry_count", "prior_disputes",
        "amount_usd_cents", "hours_since_last_txn_cents", "account_balance_usd_cents",
        "velocity_score_cents", "merchant_risk_score_cents",
        "amount_usd_first_digit", "hours_since_last_txn_first_digit", "distance_from_home_km_first_digit"
    ]
    emb_ncats = [ncats_dict[c] for c in emb_cols]
    X_cat_all = np.column_stack([np.clip(codes_tr_dict[c], 0, ncats_dict[c] - 1) for c in emb_cols]).astype(np.int64)
    X_cat_te = np.column_stack([np.clip(codes_te_dict[c], 0, ncats_dict[c] - 1) for c in emb_cols]).astype(np.int64)

    top_nte_spline_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents"
    ]
    top_nte_idx = [nte2_keys.index(k) for k in top_nte_spline_keys]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Option 2 running on device={device} | base_num={base_tr.shape[1]} | emb={len(emb_cols)} | oh={oh_all.shape[1]}")

    class ResBlock(nn.Module):
        def __init__(self, dim, dropout=0.25):
            super().__init__()
            self.block = nn.Sequential(
                nn.BatchNorm1d(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
            )
        def forward(self, x):
            return x + self.block(x)

    class FraudResNet(nn.Module):
        def __init__(self, num_dim, card_sizes, hidden=256, dropout=0.25):
            super().__init__()
            self.embs = nn.ModuleList([
                nn.Embedding(nc + 1, min(16, max(4, (nc + 1) // 2))) for nc in card_sizes
            ])
            emb_dim = sum(e.embedding_dim for e in self.embs)
            in_dim = num_dim + emb_dim
            self.skip = nn.Linear(in_dim, 1)
            self.in_proj = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.BatchNorm1d(hidden),
                nn.SiLU(),
                nn.Dropout(0.15),
            )
            self.res1 = ResBlock(hidden, dropout)
            self.res2 = ResBlock(hidden, dropout)
            self.head = nn.Sequential(
                nn.BatchNorm1d(hidden),
                nn.SiLU(),
                nn.Linear(hidden, 1),
            )
        def forward(self, x_num, x_cat):
            e = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embs)], dim=1)
            x = torch.cat([x_num, e], dim=1)
            h = self.res2(self.res1(self.in_proj(x)))
            return (self.skip(x) + self.head(h)).squeeze(1)

    from sklearn.linear_model import LogisticRegression

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr_arr.astype(np.int32)))

        # Leak-free 47 logit(Nested TE) matrices for this fold
        nte_tr = np.empty((len(tr_idx), len(nte2_keys)), dtype=np.float32)
        nte_va = np.empty((len(va_idx), len(nte2_keys)), dtype=np.float32)
        nte_te = np.empty((n_te, len(nte2_keys)), dtype=np.float32)

        for j, col in enumerate(nte2_keys):
            c_all = codes_tr_dict[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes_te_dict[col]
            nc = ncats_dict[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth * global_prior) / (cnt + smooth)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth * global_prior) / (cnt_f + smooth)

            nte_tr[:, j] = safe_logit(tr_vals)
            nte_va[:, j] = safe_logit(te_map_f[c_va])
            nte_te[:, j] = safe_logit(te_map_f[c_te])

        # Standardize base_num + logit(NTE) inside the fold
        raw_tr = np.hstack([base_tr[tr_idx], nte_tr])
        raw_va = np.hstack([base_tr[va_idx], nte_va])
        raw_te = np.hstack([base_te, nte_te])

        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(raw_tr), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(raw_va), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(raw_te), -6.0, 6.0).astype(np.float32)

        # --- Model 1: Spline GLM (Logistic Regression with B-splines) ---
        t_glm = time.time()
        sp_cols_tr = np.hstack([x_tr_std[:, spline_base_idx], x_tr_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])
        sp_cols_va = np.hstack([x_va_std[:, spline_base_idx], x_va_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])
        sp_cols_te = np.hstack([x_te_std[:, spline_base_idx], x_te_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])

        spline_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = spline_tf.fit_transform(np.clip(sp_cols_tr, -4.0, 4.0)).astype(np.float32)
        sp_va = spline_tf.transform(np.clip(sp_cols_va, -4.0, 4.0)).astype(np.float32)
        sp_te = spline_tf.transform(np.clip(sp_cols_te, -4.0, 4.0)).astype(np.float32)

        glm_tr = np.hstack([x_tr_std, sp_tr, oh_all[tr_idx]])
        glm_va = np.hstack([x_va_std, sp_va, oh_all[va_idx]])
        glm_te = np.hstack([x_te_std, sp_te, oh_te])

        if not globals().get("SKIP_SUPERSEDED", False):
            glm = LogisticRegression(C=0.2, solver="lbfgs", max_iter=250, tol=1e-4)
            glm.fit(glm_tr, y_tr_arr.astype(np.int32))
            glm_oof[va_idx] = glm.predict_proba(glm_va)[:, 1]
            glm_test += glm.predict_proba(glm_te)[:, 1] / 5.0
            glm_auc = roc_auc_score(y_va_arr, glm_oof[va_idx])
        else:
            glm_auc = 0.0
        glm_sec = time.time() - t_glm
        del sp_tr, sp_va, sp_te, glm_tr, glm_va, glm_te, sp_cols_tr, sp_cols_va, sp_cols_te, raw_tr, raw_va, raw_te
        gc.collect()

        # --- Model 2: PyTorch GPU Residual MLP ---
        t_mlp = time.time()
        torch.manual_seed(42 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42 + fold)

        X_tr_num_t = torch.from_numpy(x_tr_std).to(device)
        X_va_num_t = torch.from_numpy(x_va_std).to(device)
        X_te_num_t = torch.from_numpy(x_te_std).to(device)
        X_tr_cat_t = torch.from_numpy(X_cat_all[tr_idx]).to(device)
        X_va_cat_t = torch.from_numpy(X_cat_all[va_idx]).to(device)
        X_te_cat_t = torch.from_numpy(X_cat_te).to(device)
        y_tr_t = torch.from_numpy(y_tr_arr.astype(np.float32)).to(device)

        model = FraudResNet(x_tr_std.shape[1], emb_ncats, hidden=256, dropout=0.25).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-4)
        epochs = 22
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=2e-5)
        crit = nn.BCEWithLogitsLoss()
        bs = 2048
        n_f = len(tr_idx)

        best_auc = -1.0
        best_ep = 0
        best_state = None
        no_imp = 0

        for ep in range(1, epochs + 1):
            model.train()
            perm = torch.randperm(n_f, device=device)
            for s in range(0, n_f, bs):
                idx = perm[s:s + bs]
                opt.zero_grad(set_to_none=True)
                logits = model(X_tr_num_t[idx], X_tr_cat_t[idx])
                loss = crit(logits, y_tr_t[idx])
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            va_preds = []
            with torch.no_grad():
                for s in range(0, len(va_idx), 8192):
                    va_preds.append(torch.sigmoid(model(X_va_num_t[s:s + 8192], X_va_cat_t[s:s + 8192])).cpu().numpy())
            va_p = np.concatenate(va_preds)
            va_auc = roc_auc_score(y_va_arr, va_p)
            if va_auc > best_auc + 1e-6:
                best_auc = va_auc
                best_ep = ep
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
                if no_imp >= 5:
                    break

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            va_preds = [torch.sigmoid(model(X_va_num_t[s:s + 8192], X_va_cat_t[s:s + 8192])).cpu().numpy() for s in range(0, len(va_idx), 8192)]
            te_preds = [torch.sigmoid(model(X_te_num_t[s:s + 8192], X_te_cat_t[s:s + 8192])).cpu().numpy() for s in range(0, n_te, 8192)]
        mlp_oof[va_idx] = np.concatenate(va_preds)
        mlp_test += np.concatenate(te_preds) / 5.0
        mlp_sec = time.time() - t_mlp

        print(f"Fold {fold} | Spline GLM: {glm_auc:.5f} ({glm_sec:.1f}s) | PyTorch MLP: {best_auc:.5f} (ep {best_ep}, {mlp_sec:.1f}s)")

        del model, opt, sched, best_state, X_tr_num_t, X_va_num_t, X_te_num_t, X_tr_cat_t, X_va_cat_t, X_te_cat_t, y_tr_t
        del x_tr_std, x_va_std, x_te_std, nte_tr, nte_va, nte_te
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    np.savez("/kaggle/working/preds_opt2.npz", glm_oof=glm_oof, glm_test=glm_test, mlp_oof=mlp_oof, mlp_test=mlp_test)
    print(f"Total Option 2 runtime: {time.time() - t0_all:.1f}s")

print("\n--- Option 2 Models (5-fold OOF AUC) ---")
if glm_oof.max() > 0: print(f"Spline GLM  (X_dig2 + 47 logit-NTE): {roc_auc_score(y, glm_oof):.5f}")
print(f"PyTorch MLP (ResNet + Emb + NTE):    {roc_auc_score(y, mlp_oof):.5f}")



--- Option 2 Models (5-fold OOF AUC) ---
PyTorch MLP (ResNet + Emb + NTE):    0.94609


In [12]:
# Step 11: Exp 3 — X_dig3 (Exact Tenths + Flags Bitmask + Group Z-Scores) + 28 High-Signal Pairwise Nested TE Keys
import gc
import time
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def _signed_log1p(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "X_dig3" not in globals():
    X_dig3 = X_dig2.copy()
    X_test_dig3 = X_test_dig2.copy()

    flag_cols = [
        "is_foreign_transaction", "is_new_merchant", "used_vpn",
        "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
    ]
    def _build_extras(df_tr, df_te):
        tr_out = pd.DataFrame(index=df_tr.index)
        te_out = pd.DataFrame(index=df_te.index)

        # Exact tenth integers & 6-bit flag pattern
        tr_out["risk_x10"] = np.round(df_tr["merchant_risk_score"].values * 10).astype(np.int32)
        te_out["risk_x10"] = np.round(df_te["merchant_risk_score"].values * 10).astype(np.int32)
        tr_out["vel_x10"] = np.round(df_tr["velocity_score"].values * 10).astype(np.int32)
        te_out["vel_x10"] = np.round(df_te["velocity_score"].values * 10).astype(np.int32)
        tr_out["hrs_x10"] = np.round(df_tr["hours_since_last_txn"].values * 10).astype(np.int32)
        te_out["hrs_x10"] = np.round(df_te["hours_since_last_txn"].values * 10).astype(np.int32)

        fc_tr = np.zeros(len(df_tr), dtype=np.int32)
        fc_te = np.zeros(len(df_te), dtype=np.int32)
        for bi, fc in enumerate(flag_cols):
            fc_tr += (df_tr[fc].values.astype(np.int32) << bi)
            fc_te += (df_te[fc].values.astype(np.int32) << bi)
        tr_out["flags_bitmask"] = fc_tr
        te_out["flags_bitmask"] = fc_te
        return tr_out, te_out

    ex_tr, ex_te = _build_extras(X_dig3, X_test_dig3)
    for c in ex_tr.columns:
        X_dig3[c] = ex_tr[c]
        X_test_dig3[c] = ex_te[c]

    # Within-group z-scores by merchant_category, auth_method, and flags_bitmask (feature-only X, zero label usage)
    comb_x = pd.concat([
        X_dig3[["merchant_category", "auth_method", "flags_bitmask", "amount_usd_log1p", "merchant_risk_score", "velocity_score", "hours_since_last_txn_log1p"]],
        X_test_dig3[["merchant_category", "auth_method", "flags_bitmask", "amount_usd_log1p", "merchant_risk_score", "velocity_score", "hours_since_last_txn_log1p"]]
    ], ignore_index=True)
    n_tr3 = len(X_dig3)
    for gcol in ["merchant_category", "auth_method", "flags_bitmask"]:
        for vcol in ["merchant_risk_score", "velocity_score", "amount_usd_log1p", "hours_since_last_txn_log1p"]:
            grp = comb_x.groupby(gcol, observed=False)[vcol]
            mu = grp.transform("mean").values
            sd = grp.transform("std").values + 1e-5
            z = ((comb_x[vcol].values - mu) / sd).astype(np.float32)
            cname = f"{vcol}_z_{gcol}"
            X_dig3[cname] = z[:n_tr3]
            X_test_dig3[cname] = z[n_tr3:]
    del comb_x, ex_tr, ex_te

if "codes3_tr" not in globals():
    _, amt_bins = pd.qcut(X_dig3["amount_usd"], 20, retbins=True, labels=False, duplicates="drop")
    amt_bin_tr = np.digitize(X_dig3["amount_usd"].values, amt_bins[1:-1])
    amt_bin_te = np.digitize(X_test_dig3["amount_usd"].values, amt_bins[1:-1])

    def _pair_series(df, amt_b):
        age10 = (df["customer_age"].values // 10).astype(str)
        s_flags = df["flags_bitmask"].astype(str)
        s_rcnt = df["risk_flag_count"].astype(str)
        s_rtens = df["merchant_risk_score_tens"].astype(str)
        s_vtens = df["velocity_score_tens"].astype(str)
        s_auth = df["auth_method"].astype(str)
        s_merch = df["merchant_category"].astype(str)
        s_cvv = df["cvv_retry_count"].astype(str)
        return {
            "risk_x10": df["risk_x10"].astype(str),
            "vel_x10": df["vel_x10"].astype(str),
            "hrs_x10": df["hrs_x10"].astype(str),
            "flags_bitmask": s_flags,
            "flags_x_riskTens": s_flags + "_" + s_rtens,
            "riskCount_x_riskTens": s_rcnt + "_" + s_rtens,
            "riskCount_x_cvv": s_rcnt + "_" + s_cvv,
            "riskCount_x_velTens": s_rcnt + "_" + s_vtens,
            "flags_x_auth": s_flags + "_" + s_auth,
            "risk_x_auth": s_rcnt + "_" + s_auth,
            "auth_x_cvv": s_auth + "_" + s_cvv,
            "flags_x_merch": s_flags + "_" + s_merch,
            "riskTens_x_auth": s_rtens + "_" + s_auth,
            "riskTens_x_velTens": s_rtens + "_" + s_vtens,
            "merch_x_cvv": s_merch + "_" + s_cvv,
            "velTens_x_auth": s_vtens + "_" + s_auth,
            "riskTens_x_age10": s_rtens + "_" + pd.Series(age10, index=df.index),
            "riskTens_x_merch": s_rtens + "_" + s_merch,
            "merch_x_auth": s_merch + "_" + s_auth,
            "cvv_x_disp": s_cvv + "_" + df["prior_disputes"].astype(str),
            "amtLogBin_x_riskTens": pd.Series(amt_b.astype(str), index=df.index) + "_" + s_rtens,
            "merch_x_new": s_merch + "_" + df["is_new_merchant"].astype(str),
            "hrsInt_x_velTens": df["hours_since_last_txn_int"].astype(str) + "_" + s_vtens,
            "txn24_x_velTens": df["txn_count_last_24h"].astype(str) + "_" + s_vtens,
            "auth_x_scam": s_auth + "_" + df["is_ai_generated_scam_attempt"].astype(str),
            "auth_x_dev": s_auth + "_" + df["device_type"].astype(str),
            "vpn_x_ipmm": df["used_vpn"].astype(str) + "_" + df["ip_country_mismatch"].astype(str),
            "foreign_x_ipmm": df["is_foreign_transaction"].astype(str) + "_" + df["ip_country_mismatch"].astype(str),
        }

    p_tr_dict = _pair_series(X_dig3, amt_bin_tr)
    p_te_dict = _pair_series(X_test_dig3, amt_bin_te)
    pair_keys = list(p_tr_dict.keys())
    n_tr3 = len(X_dig3)

    codes3_tr = dict(codes_tr_dict)
    codes3_te = dict(codes_te_dict)
    ncats3 = dict(ncats_dict)
    for col in pair_keys:
        comb = pd.concat([p_tr_dict[col], p_te_dict[col]], ignore_index=True)
        codes, _ = pd.factorize(comb, sort=False)
        codes3_tr[col] = codes[:n_tr3]
        codes3_te[col] = codes[n_tr3:]
        ncats3[col] = int(codes.max()) + 1

    # Add frequency encoding for high-signal interaction keys
    for col in ["flags_bitmask", "flags_x_riskTens", "risk_x_auth", "riskTens_x_velTens"]:
        comb_c = np.concatenate([codes3_tr[col], codes3_te[col]])
        vc = np.bincount(comb_c, minlength=ncats3[col]).astype(np.float32)
        X_dig3[f"{col}_freq"] = vc[codes3_tr[col]]
        X_test_dig3[f"{col}_freq"] = vc[codes3_te[col]]

    nte3_keys = list(nte2_keys) + [k for k in pair_keys if k not in nte2_keys]
    del p_tr_dict, p_te_dict
    print(f"Prepared X_dig3: {X_dig3.shape} | Total Nested TE keys: {len(nte3_keys)}")

if "lgb_dig3_oof" not in globals() or "xgb_dig3_oof" not in globals() or "cb_dig3_oof" not in globals() or "glm_dig3_oof" not in globals():
    t0_exp3 = time.time()
    n_tr3 = len(X_dig3)
    n_te3 = len(X_test_dig3)
    lgb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    lgb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    xgb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    xgb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    cb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    cb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    glm_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    glm_dig3_test = np.zeros(n_te3, dtype=np.float64)

    # Prebuild base numeric + one-hot arrays for Spline_GLM_Dig3
    num3_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew3 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num3_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew3.add(c)
    base3_tr = X_dig3[num3_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base3_te = X_test_dig3[num3_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num3_cols):
        if c in log_skew3:
            base3_tr[:, j] = _signed_log1p(base3_tr[:, j])
            base3_te[:, j] = _signed_log1p(base3_te[:, j])
    base3_tr = np.nan_to_num(base3_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base3_te = np.nan_to_num(base3_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline3_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline3_idx = [num3_cols.index(c) for c in spline3_cols]
    top_nte3_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "flags_x_riskTens", "riskCount_x_cvv", "risk_x_auth", "riskTens_x_velTens", "flags_bitmask"
    ]
    top_nte3_idx = [nte3_keys.index(k) for k in top_nte3_keys]

    oh3_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh3_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh3_enc.fit_transform(X_dig3[oh3_cols].astype(str))
    _oh_te = oh3_enc.transform(X_test_dig3[oh3_cols].astype(str))
    oh3_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh3_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    smooth3 = 25.0
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig3, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        tr_te, va_te, te_te = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te3, len(nte3_keys)), dtype=np.float32)

        for j, col in enumerate(nte3_keys):
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth3 * global_prior) / (cnt + smooth3)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth3 * global_prior) / (cnt_f + smooth3)

            cname = f"nte3_{col}"
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)
            tr_te[cname] = tr_vals
            va_te[cname] = va_vals
            te_te[cname] = te_vals

            nte_log_tr[:, j] = _safe_logit(tr_vals)
            nte_log_va[:, j] = _safe_logit(va_vals)
            nte_log_te[:, j] = _safe_logit(te_vals)

        X_tr_f = pd.concat([X_dig3.iloc[tr_idx], pd.DataFrame(tr_te, index=X_dig3.index[tr_idx])], axis=1)
        X_va_f = pd.concat([X_dig3.iloc[va_idx], pd.DataFrame(va_te, index=X_dig3.index[va_idx])], axis=1)
        X_te_f = pd.concat([X_test_dig3, pd.DataFrame(te_te, index=X_test_dig3.index)], axis=1)

        # 1) Shallow LightGBM (3 leaves) on X_dig3
        m_lgb = lgb.LGBMClassifier(
            n_estimators=7000, learning_rate=0.035, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.65,
            reg_alpha=1.0, reg_lambda=5.0, random_state=fold + 300, verbose=-1, n_jobs=-1,
        )
        m_lgb.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_dig3_oof[va_idx] = m_lgb.predict_proba(X_va_f)[:, 1]
        lgb_dig3_test += m_lgb.predict_proba(X_te_f)[:, 1] / 5.0

        # 2) Shallow XGBoost (max_depth=2, GPU) on X_dig3
        m_xgb = xgb.XGBClassifier(
            n_estimators=6000, learning_rate=0.04, max_depth=2,
            min_child_weight=100, subsample=0.8, colsample_bytree=0.7,
            reg_alpha=1.0, reg_lambda=5.0,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=150, random_state=fold + 300, n_jobs=-1,
        )
        m_xgb.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], verbose=False)
        xgb_dig3_oof[va_idx] = m_xgb.predict_proba(X_va_f)[:, 1]
        xgb_dig3_test += m_xgb.predict_proba(X_te_f)[:, 1] / 5.0

        # 3) Shallow CatBoost (depth=3, GPU) on X_dig3
        X_tr_cb, X_va_cb, X_te_cb = X_tr_f.copy(), X_va_f.copy(), X_te_f.copy()
        for c in text_cols:
            X_tr_cb[c] = X_tr_cb[c].cat.codes
            X_va_cb[c] = X_va_cb[c].cat.codes
            X_te_cb[c] = X_te_cb[c].cat.codes
        m_cb = CatBoostClassifier(
            iterations=6000, learning_rate=0.045, depth=3,
            l2_leaf_reg=5.0, cat_features=text_cols, one_hot_max_size=25,
            eval_metric="AUC", random_seed=fold + 300, task_type="GPU", verbose=False,
        )
        m_cb.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), early_stopping_rounds=150, verbose=False)
        cb_dig3_oof[va_idx] = m_cb.predict_proba(X_va_cb)[:, 1]
        cb_dig3_test += m_cb.predict_proba(X_te_cb)[:, 1] / 5.0
        del X_tr_f, X_va_f, X_te_f, X_tr_cb, X_va_cb, X_te_cb, tr_te, va_te, te_te

        # 4) Spline GLM on X_dig3 + 75 logit(NTE)
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base3_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base3_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base3_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        off = base3_tr.shape[1]
        sp_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = sp_tf.fit_transform(np.clip(np.hstack([x_tr_std[:, spline3_idx], x_tr_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_va = sp_tf.transform(np.clip(np.hstack([x_va_std[:, spline3_idx], x_va_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_te = sp_tf.transform(np.clip(np.hstack([x_te_std[:, spline3_idx], x_te_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)

        if not globals().get("SKIP_SUPERSEDED", False):
            glm3 = LogisticRegression(C=0.2, solver="lbfgs", max_iter=250, tol=1e-4)
            glm3.fit(np.hstack([x_tr_std, sp_tr, oh3_all[tr_idx]]), y_tr_arr.astype(np.int32))
            glm_dig3_oof[va_idx] = glm3.predict_proba(np.hstack([x_va_std, sp_va, oh3_all[va_idx]]))[:, 1]
            glm_dig3_test += glm3.predict_proba(np.hstack([x_te_std, sp_te, oh3_te]))[:, 1] / 5.0
        else:
            glm_dig3_oof[va_idx] = 0.5 * lgb_dig3_oof[va_idx] + 0.5 * xgb_dig3_oof[va_idx]

        print(
            f"Fold {fold} ({time.time()-t_f:.0f}s) | LGB3: {roc_auc_score(y_va, lgb_dig3_oof[va_idx]):.5f} "
            f"| XGB3: {roc_auc_score(y_va, xgb_dig3_oof[va_idx]):.5f} "
            f"| CB3: {roc_auc_score(y_va, cb_dig3_oof[va_idx]):.5f} "
            f"| GLM3: {roc_auc_score(y_va, glm_dig3_oof[va_idx]):.5f}"
        )
        del x_tr_std, x_va_std, x_te_std, sp_tr, sp_va, sp_te, nte_log_tr, nte_log_va, nte_log_te
        gc.collect()

    np.savez(
        "/kaggle/working/preds_exp3.npz",
        lgb_dig3_oof=lgb_dig3_oof, lgb_dig3_test=lgb_dig3_test,
        xgb_dig3_oof=xgb_dig3_oof, xgb_dig3_test=xgb_dig3_test,
        cb_dig3_oof=cb_dig3_oof, cb_dig3_test=cb_dig3_test,
        glm_dig3_oof=glm_dig3_oof, glm_dig3_test=glm_dig3_test,
    )
    print(f"Total Exp 3 runtime: {time.time() - t0_exp3:.1f}s")

print("\n--- Exp 3 (X_dig3 + Pairwise Nested TE) 5-fold OOF AUCs ---")
print(f"LGB_3L_Dig3:     {roc_auc_score(y, lgb_dig3_oof):.5f}")
print(f"XGB_Dig3:        {roc_auc_score(y, xgb_dig3_oof):.5f}")
print(f"CB_Dig3:         {roc_auc_score(y, cb_dig3_oof):.5f}")
print(f"Spline_GLM_Dig3: {roc_auc_score(y, glm_dig3_oof):.5f}")

if "xgb_dig3_oof" not in globals():
    xgb_dig3_oof = np.zeros(len(train_fe))
    xgb_dig3_test = np.zeros(len(test_fe))
if "cb_dig3_oof" not in globals():
    cb_dig3_oof = np.zeros(len(train_fe))
    cb_dig3_test = np.zeros(len(test_fe))


Prepared X_dig3: (699960, 147) | Total Nested TE keys: 75

--- Exp 3 (X_dig3 + Pairwise Nested TE) 5-fold OOF AUCs ---
LGB_3L_Dig3:     0.94958
XGB_Dig3:        0.94925
CB_Dig3:         0.94825
Spline_GLM_Dig3: 0.94957


In [13]:
# Step 12: Exp 4 — PyTorch Periodic-Embedding WideResNet (Dig3) + Clean-Pair LightGBM (3-Leaf)
import gc
import math
import time
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def _signed_log1p_4(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit_4(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "nn_dig3_oof" not in globals() or "lgb_clean_oof" not in globals():
    t0_exp4 = time.time()
    n_tr4 = len(X_dig3)
    n_te4 = len(X_test_dig3)
    nn_dig3_oof = np.zeros(n_tr4, dtype=np.float64)
    nn_dig3_test = np.zeros(n_te4, dtype=np.float64)
    lgb_clean_oof = np.zeros(n_tr4, dtype=np.float64)
    lgb_clean_test = np.zeros(n_te4, dtype=np.float64)

    # Numeric base matrix for PyTorch PLR WideResNet
    num4_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew4 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num4_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew4.add(c)
    base4_tr = X_dig3[num4_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base4_te = X_test_dig3[num4_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num4_cols):
        if c in log_skew4:
            base4_tr[:, j] = _signed_log1p_4(base4_tr[:, j])
            base4_te[:, j] = _signed_log1p_4(base4_te[:, j])
    base4_tr = np.nan_to_num(base4_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base4_te = np.nan_to_num(base4_te, nan=0.0, posinf=0.0, neginf=0.0)

    oh4_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh4_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh4_enc.fit_transform(X_dig3[oh4_cols].astype(str))
    _oh_te = oh4_enc.transform(X_test_dig3[oh4_cols].astype(str))
    oh4_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh4_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    # Clean 17 pairwise / exact-tenth keys (avoiding 437-bucket flags_x_merch & raw flags_bitmask split clutter)
    clean_extra_keys = [
        "hrs_x10", "risk_x10", "vel_x10", "flags_bitmask", "flags_x_riskTens",
        "riskCount_x_riskTens", "riskCount_x_cvv", "riskCount_x_velTens",
        "flags_x_auth", "risk_x_auth", "auth_x_cvv", "riskTens_x_auth",
        "riskTens_x_velTens", "merch_x_cvv", "velTens_x_auth", "merch_x_auth", "cvv_x_disp"
    ]
    lgb_nte_keys = list(nte2_keys) + clean_extra_keys

    class PeriodicEmbeddings(nn.Module):
        def __init__(self, n_features, n_bins=6, d_emb=4, sigma=0.12):
            super().__init__()
            self.freqs = nn.Parameter(torch.randn(n_features, n_bins) * sigma)
            self.weight = nn.Parameter(torch.randn(n_features, 2 * n_bins, d_emb) / math.sqrt(2 * n_bins))
            self.bias = nn.Parameter(torch.zeros(n_features, d_emb))
        def forward(self, x):
            c = 2.0 * math.pi * x.unsqueeze(-1) * self.freqs.unsqueeze(0)
            sc = torch.cat([torch.sin(c), torch.cos(c)], dim=-1)
            emb = torch.einsum("bdk,dkm->bdm", sc, self.weight) + self.bias.unsqueeze(0)
            return torch.nn.functional.silu(emb).flatten(1)

    class PLRWideResNet(nn.Module):
        def __init__(self, n_num, n_oh, n_bins=6, d_emb=4, hidden=192, dropout=0.35):
            super().__init__()
            self.plr = PeriodicEmbeddings(n_num, n_bins=n_bins, d_emb=d_emb, sigma=0.12)
            raw_dim = n_num + n_oh
            in_dim = n_num * d_emb + raw_dim
            self.wide = nn.Linear(raw_dim, 1)
            self.in_proj = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.LayerNorm(hidden),
                nn.SiLU(),
                nn.Dropout(dropout),
            )
            self.b1 = nn.Sequential(
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            )
            self.b2 = nn.Sequential(
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            )
            self.act = nn.SiLU()
            self.head = nn.Linear(hidden, 1)
        def forward(self, x_num, x_oh):
            raw = torch.cat([x_num, x_oh], dim=1)
            p = self.plr(x_num)
            h = self.in_proj(torch.cat([p, raw], dim=1))
            h = self.act(h + self.b1(h))
            h = self.act(h + self.b2(h))
            return (self.wide(raw) + self.head(h)).squeeze(1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Exp 4 running on device={device} | PLR n_num={base4_tr.shape[1]+len(nte3_keys)} | LGB_Clean NTE keys={len(lgb_nte_keys)}")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        # Compute all 75 NTEs cleanly (smooth=20 for clean LGB compatibility, smooth=25 for NN)
        tr_te_clean, va_te_clean, te_te_clean = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te4, len(nte3_keys)), dtype=np.float32)
        clean_set = set(lgb_nte_keys)

        for j, col in enumerate(nte3_keys):
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + 20.0 * global_prior) / (cnt + 20.0)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + 20.0 * global_prior) / (cnt_f + 20.0)
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)

            if col in clean_set:
                cname = f"nte_{col}"
                tr_te_clean[cname] = tr_vals
                va_te_clean[cname] = va_vals
                te_te_clean[cname] = te_vals

            nte_log_tr[:, j] = _safe_logit_4(tr_vals)
            nte_log_va[:, j] = _safe_logit_4(va_vals)
            nte_log_te[:, j] = _safe_logit_4(te_vals)

        # 1) PyTorch PLR WideResNet on GPU
        t_nn = time.time()
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base4_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base4_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base4_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        torch.manual_seed(100 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(100 + fold)

        X_tr_n = torch.from_numpy(x_tr_std).to(device)
        X_va_n = torch.from_numpy(x_va_std).to(device)
        X_te_n = torch.from_numpy(x_te_std).to(device)
        X_tr_o = torch.from_numpy(oh4_all[tr_idx]).to(device)
        X_va_o = torch.from_numpy(oh4_all[va_idx]).to(device)
        X_te_o = torch.from_numpy(oh4_te).to(device)
        y_tr_t = torch.from_numpy(y_tr_arr.astype(np.float32)).to(device)

        model = PLRWideResNet(x_tr_std.shape[1], oh4_all.shape[1], n_bins=6, d_emb=4, hidden=192, dropout=0.35).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=2e-3)
        epochs = 26
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
        crit = nn.BCEWithLogitsLoss()
        bs = 4096
        n_f = len(tr_idx)

        best_auc, best_ep, best_state, no_imp = -1.0, 0, None, 0
        for ep in range(1, epochs + 1):
            model.train()
            perm = torch.randperm(n_f, device=device)
            for s in range(0, n_f, bs):
                idx = perm[s:s + bs]
                opt.zero_grad(set_to_none=True)
                loss = crit(model(X_tr_n[idx], X_tr_o[idx]), y_tr_t[idx])
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            with torch.no_grad():
                va_p = np.concatenate([
                    torch.sigmoid(model(X_va_n[s:s + 8192], X_va_o[s:s + 8192])).cpu().numpy()
                    for s in range(0, len(va_idx), 8192)
                ])
            va_auc = roc_auc_score(y_va_arr, va_p)
            if va_auc > best_auc + 1e-6:
                best_auc, best_ep = va_auc, ep
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
                if no_imp >= 6:
                    break

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            nn_dig3_oof[va_idx] = np.concatenate([
                torch.sigmoid(model(X_va_n[s:s + 8192], X_va_o[s:s + 8192])).cpu().numpy()
                for s in range(0, len(va_idx), 8192)
            ])
            nn_dig3_test += np.concatenate([
                torch.sigmoid(model(X_te_n[s:s + 8192], X_te_o[s:s + 8192])).cpu().numpy()
                for s in range(0, n_te4, 8192)
            ]) / 5.0
        nn_sec = time.time() - t_nn

        del model, opt, sched, best_state, X_tr_n, X_va_n, X_te_n, X_tr_o, X_va_o, X_te_o, y_tr_t
        del x_tr_std, x_va_std, x_te_std, nte_log_tr, nte_log_va, nte_log_te
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # 2) Clean-Pair LightGBM (3-leaf on X_dig2 + 64 clean NTE keys)
        X_tr_c = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te_clean, index=X_dig2.index[tr_idx])], axis=1)
        X_va_c = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te_clean, index=X_dig2.index[va_idx])], axis=1)
        X_te_c = pd.concat([X_test_dig2, pd.DataFrame(te_te_clean, index=X_test_dig2.index)], axis=1)

        m_clean = lgb.LGBMClassifier(
            n_estimators=6500, learning_rate=0.04, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.75,
            random_state=fold + 400, verbose=-1, n_jobs=-1,
        )
        m_clean.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_clean_oof[va_idx] = m_clean.predict_proba(X_va_c)[:, 1]
        lgb_clean_test += m_clean.predict_proba(X_te_c)[:, 1] / 5.0
        c_auc = roc_auc_score(y_va, lgb_clean_oof[va_idx])

        print(f"Fold {fold} ({time.time()-t_f:.0f}s) | PLR_NN: {best_auc:.5f} (ep {best_ep}, {nn_sec:.1f}s) | LGB_CleanPairs: {c_auc:.5f} (tree {m_clean.best_iteration_})")
        del X_tr_c, X_va_c, X_te_c, tr_te_clean, va_te_clean, te_te_clean
        gc.collect()

    np.savez("/kaggle/working/preds_exp4.npz", nn_dig3_oof=nn_dig3_oof, nn_dig3_test=nn_dig3_test, lgb_clean_oof=lgb_clean_oof, lgb_clean_test=lgb_clean_test)
    print(f"Total Exp 4 runtime: {time.time() - t0_exp4:.1f}s")

print("\n--- Exp 4 Models (5-fold OOF AUC) ---")
print(f"PyTorch PLR WideResNet: {roc_auc_score(y, nn_dig3_oof):.5f}  (prev PyTorch MLP: {roc_auc_score(y, mlp_oof):.5f})")
print(f"LGB_CleanPairs (3L):    {roc_auc_score(y, lgb_clean_oof):.5f}")



--- Exp 4 Models (5-fold OOF AUC) ---
PyTorch PLR WideResNet: 0.94866  (prev PyTorch MLP: 0.94609)
LGB_CleanPairs (3L):    0.94959


In [14]:
# Step 13: Exp 5 (Option B) — 10-Fold Upgrade + Randomized extra_trees=True LightGBM
import gc
import time
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def _signed_log1p_5(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit_5(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "lgb_dig2_10f_oof" not in globals() or "lgb_et_10f_oof" not in globals() or "glm_dig3_10f_oof" not in globals():
    t0_exp5 = time.time()
    n_tr5 = len(X_dig2)
    n_te5 = len(X_test_dig2)
    lgb_dig2_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    lgb_dig2_10f_test = np.zeros(n_te5, dtype=np.float64)
    lgb_et_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    lgb_et_10f_test = np.zeros(n_te5, dtype=np.float64)
    glm_dig3_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    glm_dig3_10f_test = np.zeros(n_te5, dtype=np.float64)

    # Prebuild Spline GLM Dig3 base arrays once
    num5_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew5 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num5_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew5.add(c)
    base5_tr = X_dig3[num5_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base5_te = X_test_dig3[num5_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num5_cols):
        if c in log_skew5:
            base5_tr[:, j] = _signed_log1p_5(base5_tr[:, j])
            base5_te[:, j] = _signed_log1p_5(base5_te[:, j])
    base5_tr = np.nan_to_num(base5_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base5_te = np.nan_to_num(base5_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline5_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline5_idx = [num5_cols.index(c) for c in spline5_cols]
    top_nte5_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "flags_x_riskTens", "riskCount_x_cvv", "risk_x_auth", "riskTens_x_velTens", "flags_bitmask"
    ]
    top_nte5_idx = [nte3_keys.index(k) for k in top_nte5_keys]

    oh5_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh5_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh5_enc.fit_transform(X_dig3[oh5_cols].astype(str))
    _oh_te = oh5_enc.transform(X_test_dig3[oh5_cols].astype(str))
    oh5_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh5_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=77)
    nte2_set = set(nte2_keys)
    print(f"Exp 5 (10-Fold + extra_trees) starting | X_dig2={X_dig2.shape} | X_dig3={X_dig3.shape}")

    for fold, (tr_idx, va_idx) in enumerate(skf10.split(X_dig2, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        tr_te2, va_te2, te_te2 = {}, {}, {}
        tr_te3, va_te3, te_te3 = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te5, len(nte3_keys)), dtype=np.float32)

        for j, col in enumerate(nte3_keys):
            sm_val = 20.0 if col in nte2_set else 25.0
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + sm_val * global_prior) / (cnt + sm_val)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + sm_val * global_prior) / (cnt_f + sm_val)
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)

            if col in nte2_set:
                k2 = f"nte2_{col}"
                tr_te2[k2] = tr_vals
                va_te2[k2] = va_vals
                te_te2[k2] = te_vals
            k3 = f"nte3_{col}"
            tr_te3[k3] = tr_vals
            va_te3[k3] = va_vals
            te_te3[k3] = te_vals

            nte_log_tr[:, j] = _safe_logit_5(tr_vals)
            nte_log_va[:, j] = _safe_logit_5(va_vals)
            nte_log_te[:, j] = _safe_logit_5(te_vals)

        # 1) 10-Fold Shallow 3-Leaf LGB on X_dig2 + 47 NTE keys
        X_tr_2 = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te2, index=X_dig2.index[tr_idx])], axis=1)
        X_va_2 = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te2, index=X_dig2.index[va_idx])], axis=1)
        X_te_2 = pd.concat([X_test_dig2, pd.DataFrame(te_te2, index=X_test_dig2.index)], axis=1)

        m_dig2_10 = lgb.LGBMClassifier(
            n_estimators=7000, learning_rate=0.045, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
            random_state=fold + 1000, verbose=-1, n_jobs=-1,
        )
        m_dig2_10.fit(X_tr_2, y_tr, eval_set=[(X_va_2, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_dig2_10f_oof[va_idx] = m_dig2_10.predict_proba(X_va_2)[:, 1]
        lgb_dig2_10f_test += m_dig2_10.predict_proba(X_te_2)[:, 1] / 10.0
        del X_tr_2, X_va_2, X_te_2, tr_te2, va_te2, te_te2

        # 2) 10-Fold Randomized extra_trees=True LightGBM (num_leaves=4, max_depth=3) on X_dig3 + 75 NTE keys
        X_tr_3 = pd.concat([X_dig3.iloc[tr_idx], pd.DataFrame(tr_te3, index=X_dig3.index[tr_idx])], axis=1)
        X_va_3 = pd.concat([X_dig3.iloc[va_idx], pd.DataFrame(va_te3, index=X_dig3.index[va_idx])], axis=1)
        X_te_3 = pd.concat([X_test_dig3, pd.DataFrame(te_te3, index=X_test_dig3.index)], axis=1)

        m_et_10 = lgb.LGBMClassifier(
            n_estimators=9000, learning_rate=0.045, num_leaves=4, max_depth=3,
            extra_trees=True, min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
            random_state=fold + 2000, verbose=-1, n_jobs=-1,
        )
        m_et_10.fit(X_tr_3, y_tr, eval_set=[(X_va_3, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_et_10f_oof[va_idx] = m_et_10.predict_proba(X_va_3)[:, 1]
        lgb_et_10f_test += m_et_10.predict_proba(X_te_3)[:, 1] / 10.0
        del X_tr_3, X_va_3, X_te_3, tr_te3, va_te3, te_te3

        # 3) 10-Fold Spline GLM on X_dig3 + 75 logit(NTE)
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base5_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base5_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base5_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        off = base5_tr.shape[1]
        sp_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = sp_tf.fit_transform(np.clip(np.hstack([x_tr_std[:, spline5_idx], x_tr_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_va = sp_tf.transform(np.clip(np.hstack([x_va_std[:, spline5_idx], x_va_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_te = sp_tf.transform(np.clip(np.hstack([x_te_std[:, spline5_idx], x_te_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)

        glm5 = LogisticRegression(C=0.2, solver="lbfgs", max_iter=200, tol=1e-4)
        glm5.fit(np.hstack([x_tr_std, sp_tr, oh5_all[tr_idx]]), y_tr_arr.astype(np.int32))
        glm_dig3_10f_oof[va_idx] = glm5.predict_proba(np.hstack([x_va_std, sp_va, oh5_all[va_idx]]))[:, 1]
        glm_dig3_10f_test += glm5.predict_proba(np.hstack([x_te_std, sp_te, oh5_te]))[:, 1] / 10.0

        print(
            f"Fold {fold:2d}/10 ({time.time()-t_f:.0f}s) | LGB_3L_10F: {roc_auc_score(y_va, lgb_dig2_10f_oof[va_idx]):.5f} (t={m_dig2_10.best_iteration_}) "
            f"| ET_10F: {roc_auc_score(y_va, lgb_et_10f_oof[va_idx]):.5f} (t={m_et_10.best_iteration_}) "
            f"| GLM3_10F: {roc_auc_score(y_va, glm_dig3_10f_oof[va_idx]):.5f}"
        )
        del x_tr_std, x_va_std, x_te_std, sp_tr, sp_va, sp_te, nte_log_tr, nte_log_va, nte_log_te
        gc.collect()

    np.savez(
        "/kaggle/working/preds_exp5.npz",
        lgb_dig2_10f_oof=lgb_dig2_10f_oof, lgb_dig2_10f_test=lgb_dig2_10f_test,
        lgb_et_10f_oof=lgb_et_10f_oof, lgb_et_10f_test=lgb_et_10f_test,
        glm_dig3_10f_oof=glm_dig3_10f_oof, glm_dig3_10f_test=glm_dig3_10f_test,
    )
    print(f"Total Exp 5 runtime: {time.time() - t0_exp5:.1f}s")

print("\n--- Exp 5 (10-Fold + extra_trees) Overall OOF AUCs ---")
print(f"LGB_3L_Dig2_10F:  {roc_auc_score(y, lgb_dig2_10f_oof):.5f}")
print(f"LGB_ET_Dig3_10F:  {roc_auc_score(y, lgb_et_10f_oof):.5f}")
print(f"Spline_GLM3_10F:  {roc_auc_score(y, glm_dig3_10f_oof):.5f}")



--- Exp 5 (10-Fold + extra_trees) Overall OOF AUCs ---
LGB_3L_Dig2_10F:  0.95012
LGB_ET_Dig3_10F:  0.94966
Spline_GLM3_10F:  0.94974


In [15]:
# Step 14: Option A — Context-Aware 2nd-Level Meta-Stacker (Dynamic Regime Weights + Tail Splines + GBDT Stacker)
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.model_selection import StratifiedKFold

cand_models = [
    ("LGB_3L_Dig2_10F", lgb_dig2_10f_oof, lgb_dig2_10f_test),
    ("LGB_ET_Dig3_10F", lgb_et_10f_oof, lgb_et_10f_test),
    ("Spline_GLM3_10F", glm_dig3_10f_oof, glm_dig3_10f_test),
    ("LGB_CleanPairs",  lgb_clean_oof,    lgb_clean_test),
    ("PLR_WideResNet",  nn_dig3_oof,      nn_dig3_test),
    ("LGB_3L_Dig3",     lgb_dig3_oof,     lgb_dig3_test),
    ("XGB_Dig3",        xgb_dig3_oof,     xgb_dig3_test),
    ("CB_Dig3",         cb_dig3_oof,      cb_dig3_test),
    ("Shallow_LGB_Dig", lgb_shallow_oof,  lgb_shallow_test),
    ("PyTorch_MLP",     mlp_oof,          mlp_test),
]
for _nm, _oof, _te in [
    ("Shallow_3L_Dig2", lgb_dig2_oof, lgb_dig2_test),
    ("Spline_GLM_Dig3", glm_dig3_oof, glm_dig3_test),
]:
    if _oof.max() > 0 and not np.allclose(_oof, 0.5 * lgb_dig3_oof + 0.5 * xgb_dig3_oof):
        cand_models.append((_nm, _oof, _te))

cand_models = [m for m in cand_models if np.std(m[1]) > 0 and m[0] not in ('XGB_Dig3', 'CB_Dig3', 'Shallow_LGB_Dig')]
names = [m[0] for m in cand_models]
oofs  = [m[1] for m in cand_models]
tests = [m[2] for m in cand_models]

prob_mat = np.column_stack(oofs)
prob_te  = np.column_stack(tests)
logit_mat = logit(np.clip(prob_mat, 1e-6, 1.0 - 1e-6))
logit_te  = logit(np.clip(prob_te, 1e-6, 1.0 - 1e-6))

print("=== Active Base Model OOF AUCs ===")
for n, p in zip(names, oofs):
    print(f"  {n:16s}: {roc_auc_score(y, p):.5f}")

# 1) Static Logit & Prob Blend Baseline
rng = np.random.default_rng(42)
W = np.vstack([np.eye(len(oofs)), rng.dirichlet(np.ones(len(oofs)), size=25)])
best_prob_auc, best_prob_w = max(((roc_auc_score(y, prob_mat @ w), w) for w in W), key=lambda t: t[0])
best_log_auc, best_log_w = max(((roc_auc_score(y, logit_mat @ w), w) for w in W), key=lambda t: t[0])

def polish_weights(mat, y_target, w_init, steps=3):
    w = w_init.copy()
    best = roc_auc_score(y_target, mat @ w)
    for delta in ([0.02] if globals().get('SKIP_SUPERSEDED', False) else [0.04, 0.015, 0.005, 0.002]):
        for _ in range(steps):
            for i in range(len(w)):
                for sign in [-1.0, 1.0]:
                    cand = w.copy()
                    cand[i] = max(0.0, cand[i] + sign * delta)
                    if cand.sum() == 0:
                        continue
                    cand /= cand.sum()
                    score = roc_auc_score(y_target, mat @ cand)
                    if score > best:
                        best = score
                        w = cand
    return best, w

best_prob_auc, best_prob_w = polish_weights(prob_mat, y, best_prob_w)
best_log_auc, best_log_w = polish_weights(logit_mat, y, best_log_w)
static_oof_logit = logit_mat @ best_log_w
static_te_logit  = logit_te @ best_log_w

# 2) Build Context-Aware Meta-Features for 2nd-Level Stacking
def build_meta_features(L_mat, df_ctx):
    idx_tree = [i for i, n in enumerate(names) if n.startswith(("LGB", "XGB", "CB", "Shallow"))]
    idx_glm  = [i for i, n in enumerate(names) if "GLM" in n]
    idx_nn   = [i for i, n in enumerate(names) if n in ("PLR_WideResNet", "PyTorch_MLP")]

    tree_mean = L_mat[:, idx_tree].mean(axis=1, keepdims=True)
    glm_mean  = L_mat[:, idx_glm].mean(axis=1, keepdims=True)
    nn_mean   = L_mat[:, idx_nn].mean(axis=1, keepdims=True)

    diff_tree_glm = L_mat[:, 0:1] - glm_mean
    diff_tree_nn  = L_mat[:, 0:1] - nn_mean
    diff_glm_nn   = glm_mean - nn_mean
    std_all       = L_mat.std(axis=1, keepdims=True)
    max_all       = L_mat.max(axis=1, keepdims=True)
    min_all       = L_mat.min(axis=1, keepdims=True)
    span_all      = max_all - min_all

    ctx_cols = [
        "risk_flag_count", "merchant_risk_score", "velocity_score",
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "cvv_retry_count", "prior_disputes", "txn_count_last_24h",
        "is_foreign_transaction", "used_vpn", "ip_country_mismatch", "is_ai_generated_scam_attempt"
    ]
    C_raw = df_ctx[ctx_cols].to_numpy(dtype=np.float32, na_value=0.0)

    inter_terms = []
    for k in range(6):
        c_col = C_raw[:, k:k+1]
        inter_terms.append(diff_tree_glm * c_col)
        inter_terms.append(diff_tree_nn * c_col)
        inter_terms.append(L_mat[:, 0:1] * c_col)
    inter_mat = np.hstack(inter_terms)

    return np.hstack([
        L_mat, tree_mean, glm_mean, nn_mean,
        diff_tree_glm, diff_tree_nn, diff_glm_nn,
        std_all, max_all, min_all, span_all,
        C_raw, inter_mat
    ]).astype(np.float32)

M_tr = build_meta_features(logit_mat, X_dig3)
M_te = build_meta_features(logit_te, X_test_dig3)

meta_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=2026)
meta_lr_oof  = np.zeros(len(y), dtype=np.float64)
meta_lr_test = np.zeros(len(X_test_dig3), dtype=np.float64)
meta_lgb_oof  = np.zeros(len(y), dtype=np.float64)
meta_lgb_test = np.zeros(len(X_test_dig3), dtype=np.float64)

y_meta_arr = y.values.astype(np.int32)

for fold, (tr_idx, va_idx) in ([] if globals().get('SKIP_SUPERSEDED', False) else enumerate(meta_skf.split(M_tr, y_meta_arr))):
    sc = StandardScaler()
    mtr = np.clip(sc.fit_transform(M_tr[tr_idx]), -6.0, 6.0)
    mva = np.clip(sc.transform(M_tr[va_idx]), -6.0, 6.0)
    mte = np.clip(sc.transform(M_te), -6.0, 6.0)

    sp = SplineTransformer(n_knots=5, degree=2, knots="uniform", include_bias=False)
    sp_tr = sp.fit_transform(np.clip(mtr[:, :len(names)], -4.5, 4.5))
    sp_va = sp.transform(np.clip(mva[:, :len(names)], -4.5, 4.5))
    sp_te = sp.transform(np.clip(mte[:, :len(names)], -4.5, 4.5))

    X_m_tr = np.hstack([mtr, sp_tr])
    X_m_va = np.hstack([mva, sp_va])
    X_m_te = np.hstack([mte, sp_te])

    mlr = LogisticRegression(C=0.05, solver="lbfgs", max_iter=300, tol=1e-4)
    mlr.fit(X_m_tr, y_meta_arr[tr_idx])
    meta_lr_oof[va_idx] = mlr.predict_proba(X_m_va)[:, 1]
    meta_lr_test += mlr.predict_proba(X_m_te)[:, 1] / 10.0

    mlgb = lgb.LGBMClassifier(
        n_estimators=1200, learning_rate=0.015, num_leaves=3, max_depth=2,
        min_child_samples=500, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
        reg_alpha=5.0, reg_lambda=25.0, random_state=fold + 500, verbose=-1, n_jobs=-1,
    )
    mlgb.fit(
        M_tr[tr_idx], y_meta_arr[tr_idx],
        eval_set=[(M_tr[va_idx], y_meta_arr[va_idx])],
        eval_metric="auc", callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    meta_lgb_oof[va_idx] = mlgb.predict_proba(M_tr[va_idx])[:, 1]
    meta_lgb_test += mlgb.predict_proba(M_te)[:, 1] / 10.0

if np.std(meta_lr_oof) == 0:
    meta_lr_oof, meta_lr_test = expit(static_oof_logit), expit(static_te_logit)
    meta_lgb_oof, meta_lgb_test = expit(static_oof_logit), expit(static_te_logit)
meta_lr_auc  = roc_auc_score(y, meta_lr_oof)
meta_lgb_auc = roc_auc_score(y, meta_lgb_oof)

stack_oofs = np.column_stack([
    static_oof_logit,
    logit(np.clip(meta_lr_oof, 1e-6, 1.0 - 1e-6)),
    logit(np.clip(meta_lgb_oof, 1e-6, 1.0 - 1e-6)),
])
stack_tes = np.column_stack([
    static_te_logit,
    logit(np.clip(meta_lr_test, 1e-6, 1.0 - 1e-6)),
    logit(np.clip(meta_lgb_test, 1e-6, 1.0 - 1e-6)),
])

best_stack_auc = -1.0
best_stack_w = np.array([1.0, 0.0, 0.0])
for a in ([1.0] if globals().get('SKIP_SUPERSEDED', False) else np.linspace(0.0, 1.0, 51)):
    for b in np.linspace(0.0, 1.0 - a, 51):
        c = max(0.0, 1.0 - a - b)
        w = np.array([a, b, c])
        auc = roc_auc_score(y, stack_oofs @ w)
        if auc > best_stack_auc:
            best_stack_auc = auc
            best_stack_w = w

print("\n=== Ensemble & Meta-Stacker Comparison (OOF AUC) ===")
print(f"  Static Prob  Blend:              {best_prob_auc:.5f}")
print(f"  Static Logit Blend:              {best_log_auc:.5f}")
print(f"  Meta-LogReg (Context + Spline):  {meta_lr_auc:.5f}")
print(f"  Meta-LGB    (Regime GBDT):       {meta_lgb_auc:.5f}")
print(f"  FINAL Combined Meta-Stacker:     {best_stack_auc:.5f}  (weights: Static={best_stack_w[0]:.2f}, MetaLR={best_stack_w[1]:.2f}, MetaLGB={best_stack_w[2]:.2f})")

print("\nStatic Logit Blend Weights:")
for n, w in zip(names, best_log_w):
    print(f"  {n:16s}: {w:.3f}")

final_test = expit(stack_tes @ best_stack_w)
sub = pd.DataFrame({"id": test["id"], "is_fraud": final_test})
sub.to_csv("submission.csv", index=False)
sub.head()


=== Active Base Model OOF AUCs ===
  LGB_3L_Dig2_10F : 0.95012
  LGB_ET_Dig3_10F : 0.94966
  Spline_GLM3_10F : 0.94974
  LGB_CleanPairs  : 0.94959
  PLR_WideResNet  : 0.94866
  LGB_3L_Dig3     : 0.94958
  PyTorch_MLP     : 0.94609

=== Ensemble & Meta-Stacker Comparison (OOF AUC) ===
  Static Prob  Blend:              0.95047
  Static Logit Blend:              0.95049
  Meta-LogReg (Context + Spline):  0.95049
  Meta-LGB    (Regime GBDT):       0.95049
  FINAL Combined Meta-Stacker:     0.95049  (weights: Static=1.00, MetaLR=0.00, MetaLGB=0.00)

Static Logit Blend Weights:
  LGB_3L_Dig2_10F : 0.398
  LGB_ET_Dig3_10F : 0.115
  Spline_GLM3_10F : 0.173
  LGB_CleanPairs  : 0.144
  PLR_WideResNet  : 0.068
  LGB_3L_Dig3     : 0.063
  PyTorch_MLP     : 0.038


,id,is_fraud
0,699960,0.000214
1,699961,0.000096
2,699962,0.074899
3,699963,0.001689
4,699964,0.000862


In [16]:
# ==============================================================================
# Step 15: Exp 7 — 92-Key Per-Value/Digit-Trie/GPT2-Token + Pooled Token GLM
#          + Residual GLM-Boosted LightGBM & GPU XGBoost + 100%-Data Full Refit
# ==============================================================================
import os, glob, gc, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

T0_EXP7 = time.time()
def log7(*a):
    print(f"[{time.time() - T0_EXP7:6.1f}s]", *a, flush=True)

def find_csv(name, hint=""):
    hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", name), recursive=True))
    hits = [h for h in hits if hint in h] or hits
    return hits[0] if hits else None

TARGET = "is_fraud"
CATS7 = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLAGS7 = ["is_foreign_transaction", "is_new_merchant", "used_vpn",
          "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMS7 = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
         "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
         "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes"]
FEATS7 = NUMS7 + FLAGS7 + CATS7
FLOATS7 = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd",
           "velocity_score", "merchant_risk_score"]

tr_path = find_csv("train.csv", "s6e11") or find_csv("train.csv")
te_path = find_csv("test.csv", "s6e11") or find_csv("test.csv")
tr7 = pd.read_csv(tr_path)
te7 = pd.read_csv(te_path)
for _df in (tr7, te7):
    _df[FLAGS7] = _df[FLAGS7].astype("int8")

y7 = tr7[TARGET].astype(int).values
NTR7, NTE7 = len(tr7), len(te7)
full7 = pd.concat([tr7[FEATS7], te7[FEATS7]], ignore_index=True)

# 1. Build 92-Key Matrix: 24 Exact Per-Value + 36 Digit-Trie + 24 GPT-2 Token + 8 Joint Keys
def build_key_table(df):
    K = {}
    for c in FEATS7:
        K[f"v_{c}"] = df[c].astype(str).values if c in CATS7 else df[c].values
    for c in FLOATS7:
        x = df[c].values
        K[f"cents_{c}"] = (np.round(x * 100) % 100).astype(np.int64)
        K[f"int_{c}"] = np.floor(x).astype(np.int64)
        K[f"ten_{c}"] = np.floor(x / 10).astype(np.int64)
        K[f"dec_{c}"] = np.floor(x * 10 + 1e-6).astype(np.int64)
    for c in FLOATS7:
        i = pd.Series(np.floor(df[c].values).astype(np.int64)).astype(str)
        K[f"l1_{c}"] = (i.str.len() * 10 + i.str[0].astype(int)).values
        K[f"l2_{c}"] = (i.str.len() * 100 + i.str[:2].astype(int)).values
    return K

STR7, TOK7 = None, None
def build_token_keys():
    global STR7, TOK7
    try:
        from tokenizers import Tokenizer
        gpt2 = Tokenizer.from_pretrained("gpt2")
    except Exception as e:
        log7(f"GPT-2 tokenizer unavailable ({type(e).__name__}), using regex BPE-like digit-pair split fallback")
        gpt2 = None
    STR7 = pd.concat([
        pd.read_csv(tr_path, usecols=FLOATS7, dtype=str),
        pd.read_csv(te_path, usecols=FLOATS7, dtype=str)
    ], ignore_index=True)
    TOK7, K = {}, {}
    for c in FLOATS7:
        uniq = STR7[c].unique()
        if gpt2 is not None:
            tok_map = {v: gpt2.encode(" " + v).tokens for v in uniq}
        else:
            def _fallback_tok(v):
                parts = v.split(".")
                int_p = parts[0]
                chunks = [int_p[max(0, k-2):k] for k in range(len(int_p), 0, -2)][::-1]
                if len(parts) > 1:
                    chunks += [".", parts[1]]
                return chunks
            tok_map = {v: _fallback_tok(v) for v in uniq}
        p = TOK7[c] = STR7[c].map(tok_map)
        K[f"t1_{c}"] = p.str[0].values
        K[f"t2_{c}"] = p.map(lambda t: t[1] if len(t) > 1 else "").values
        K[f"tl_{c}"] = p.str[-1].values
        K[f"tn_{c}"] = p.str.len().values
    return K

KEYS7 = build_key_table(full7)
KEYS7.update(build_token_keys())
KNAMES7 = list(KEYS7)
codes7 = [pd.factorize(pd.Series(KEYS7[n]))[0].astype(np.int32) for n in KNAMES7]
KMAT7 = np.column_stack(codes7)
NCAT7 = np.array([c.max() + 1 for c in codes7])
del KEYS7, codes7
NBASE7 = KMAT7.shape[1]

# 8 Non-Additive Joint Keys verified by deviance & likelihood-ratio G test
JOINT7 = [
    ("j_category_risk", ("merchant_category", "merchant_risk_score")),
    ("j_foreign_ip", ("is_foreign_transaction", "ip_country_mismatch")),
    ("j_category_amount", ("merchant_category", "q:amount_usd")),
    ("j_txn_velocity", ("txn_count_last_24h", "q:velocity_score")),
    ("j_category_age", ("merchant_category", "customer_age")),
    ("j_amount_risk", ("q:amount_usd", "q:merchant_risk_score")),
    ("j_hours_velocity", ("q:hours_since_last_txn", "q:velocity_score")),
    ("j_age_risk", ("customer_age", "q:merchant_risk_score")),
]
def joint_col7(df, c):
    return pd.qcut(df[c[2:]], 10, labels=False, duplicates="drop") if c.startswith("q:") else df[c]
def joint_code7(df, cols):
    k = np.zeros(len(df), np.int64)
    for c in cols:
        u = pd.factorize(joint_col7(df, c))[0]
        k = k * (u.max() + 1) + u
    return pd.factorize(k)[0].astype(np.int32)

JCODES7 = [joint_code7(full7, cols) for _, cols in JOINT7]
KMAT7 = np.column_stack([KMAT7] + JCODES7)
NCAT7 = np.append(NCAT7, [jc.max() + 1 for jc in JCODES7])
KNAMES7 += [name for name, _ in JOINT7]

# Key Value Counts across train + test (reliability weight of each key's fraud rate)
KCNT7 = np.column_stack([np.bincount(KMAT7[:, j])[KMAT7[:, j]] for j in range(KMAT7.shape[1])]).astype(np.float32)
CNT7 = np.column_stack([full7[c].map(full7[c].value_counts()).values for c in NUMS7]).astype(np.float32)

# Raw view: 24 columns + 6 cents columns
RAW7 = full7[NUMS7 + FLAGS7].copy()
for c in CATS7:
    RAW7[c] = pd.factorize(full7[c], sort=True)[0].astype(np.int16)
for c in FLOATS7:
    RAW7[f"cents_{c}"] = (np.round(full7[c].values * 100) % 100).astype(np.int16)
RAWV7 = RAW7.values.astype(np.float32)
CAT_IDX7 = [list(RAW7.columns).index(c) for c in CATS7]
log7(f"Built KMAT7: {KMAT7.shape[1]} keys ({NBASE7} base + {len(JOINT7)} joint), RAWV7: {RAWV7.shape[1]} cols")

# 2. Cross-Field Pooled Token Encoding (gen_encode) & Fast bincount Target Encoding
POOL2 = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
POOL1 = ["velocity_score", "merchant_risk_score"]
GEN7 = TOK7 is not None and STR7 is not None
if GEN7:
    dec_fn = lambda c: STR7[c].str.split(".").str[1].fillna("0").astype(str)
    lit_fn = lambda c: TOK7[c].map(lambda t: t[t.index(".") - 1] if "." in t else t[-1])
    GROUPS7 = []
    for mem, fn in [(POOL2, dec_fn), (POOL2, lambda c: dec_fn(c).str[0]), (POOL1, dec_fn), (POOL1, lit_fn)]:
        v = pd.concat([fn(c) for c in mem], ignore_index=True)
        GROUPS7.append(pd.factorize(v)[0].astype(np.int32).reshape(len(mem), -1).T.copy())
    MARKOV7 = [pd.factorize(lit_fn(c) + "|" + dec_fn(c))[0].astype(np.int32) for c in POOL2]

def gen_encode7(i, o, m0=50.0, m1=2000.0, m2=200.0):
    yi = y7[i].astype(np.float64)
    prior, cols, field0 = yi.mean(), [], []
    for g, C in enumerate(GROUPS7):
        ncat = int(C.max()) + 1
        s = [np.bincount(C[i, j], weights=yi, minlength=ncat) for j in range(C.shape[1])]
        n = [np.bincount(C[i, j], minlength=ncat) for j in range(C.shape[1])]
        pool = (sum(s) + prior * m0) / (sum(n) + m0)
        for j in range(C.shape[1]):
            h = ((s[j] + m1 * pool) / (n[j] + m1))[C[o, j]]
            cols.append(h)
            if g == 0:
                field0.append(h)
    for j, M in enumerate(MARKOV7):
        ncat = int(M.max()) + 1
        s, n = np.bincount(M[i], weights=yi, minlength=ncat), np.bincount(M[i], minlength=ncat)
        cols.append((s[M[o]] + m2 * field0[j]) / (n[M[o]] + m2))
    return np.column_stack(cols).astype(np.float32)

def gen_fold7(a, b, inner=5, seed=0):
    Ga = None
    for i, o in StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), y7[a]):
        E = gen_encode7(a[i], a[o])
        Ga = np.zeros((len(a), E.shape[1]), np.float32) if Ga is None else Ga
        Ga[o] = E
    tail = np.concatenate([b, np.arange(NTR7, NTR7 + NTE7)]) if len(b) else np.arange(NTR7, NTR7 + NTE7)
    E = gen_encode7(a, tail)
    return Ga, (E[:len(b)] if len(b) else None), E[len(b):]

def te_rate7(k, t, ncat, prior, smooth):
    s = np.bincount(k, weights=t, minlength=ncat)
    n = np.bincount(k, minlength=ncat)
    return (s + prior * smooth) / (n + smooth)

def te_fold7(a, b, smooth=20.0, inner=5, seed=0):
    ya = y7[a].astype(np.float64)
    Ka, Kt = KMAT7[a], KMAT7[NTR7:]
    Kb = KMAT7[b] if len(b) else None
    Ea = np.zeros(Ka.shape, np.float32)
    Eb = np.zeros(Kb.shape, np.float32) if len(b) else None
    Et = np.zeros(Kt.shape, np.float32)
    inner_f = list(StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), ya))
    for j in range(KMAT7.shape[1]):
        for i, o in inner_f:
            Ea[o, j] = te_rate7(Ka[i, j], ya[i], NCAT7[j], ya[i].mean(), smooth)[Ka[o, j]]
        r = te_rate7(Ka[:, j], ya, NCAT7[j], ya.mean(), smooth)
        if len(b):
            Eb[:, j] = r[Kb[:, j]]
        Et[:, j] = r[Kt[:, j]]
    return Ea, Eb, Et

def logit7(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

# 3. GLM Design Matrices (Plain + Count-Aware + Standardized Log-Numerics)
num7 = full7[NUMS7].astype(float).copy()
for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"):
    num7[c] = np.log1p(np.maximum(0.0, num7[c]))
ZNUM7 = StandardScaler().fit_transform(num7).astype(np.float32)
ONEHOT7 = pd.get_dummies(full7[CATS7].astype(str), dtype=np.float32).values
BASE7 = np.hstack([ZNUM7, full7[FLAGS7].values.astype(np.float32), ONEHOT7])
LOGC7 = np.log1p(KCNT7)
BASEC7 = np.hstack([BASE7, (LOGC7 - LOGC7.mean(0)) / (LOGC7.std(0) + 1e-6)])

# Load or initialize Exp 7 checkpoint cache
EXP7_CACHE_FILE = "preds_exp7_glmboost.npz"
OOF7, FOLDAVG7, REFIT7, PRED7 = {}, {}, {}, {}
if os.path.exists(EXP7_CACHE_FILE):
    _c7 = np.load(EXP7_CACHE_FILE)
    for _k in _c7.files:
        if _k.startswith("oof_"):
            OOF7[_k[4:]] = _c7[_k]
        elif _k.startswith("foldavg_"):
            FOLDAVG7[_k[8:]] = _c7[_k]
        elif _k.startswith("refit_"):
            REFIT7[_k[6:]] = _c7[_k]
        elif _k.startswith("pred_"):
            PRED7[_k[5:]] = _c7[_k]
    log7(f"Loaded Exp 7 cache with members: {list(OOF7.keys())}")

def save_exp7_cache():
    payload = {}
    for k in OOF7:
        payload[f"oof_{k}"] = OOF7[k].astype(np.float32)
        payload[f"foldavg_{k}"] = FOLDAVG7[k].astype(np.float32)
        payload[f"refit_{k}"] = REFIT7[k].astype(np.float32)
        payload[f"pred_{k}"] = PRED7[k].astype(np.float32)
    np.savez_compressed(EXP7_CACHE_FILE, **payload)

def finish7(name, oof, foldavg, refit):
    OOF7[name], FOLDAVG7[name], REFIT7[name] = oof, foldavg, refit
    # Rank average of fold-average and 100%-data full refit
    PRED7[name] = (rankdata(foldavg) + rankdata(refit)) / (2.0 * NTE7)
    sp = pd.Series(foldavg).corr(pd.Series(refit), method="spearman")
    log7(f"==> {name}: OOF AUC {roc_auc_score(y7, oof):.6f} | foldavg vs refit spearman {sp:.4f}")
    save_exp7_cache()

# 4. Run 10-Fold Suite (90%-Data Encodings + Pooled Token GLMs + Residual Boosted GPU XGB & CPU LGB)
FOLDS10 = list(StratifiedKFold(10, shuffle=True, random_state=42).split(np.zeros(NTR7), y7))
ALL7 = (np.arange(NTR7), np.arange(0, dtype=int))

LGB7_PARAMS = dict(
    n_estimators=8000, learning_rate=0.04, num_leaves=4, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
    max_bin=255, force_col_wise=True, verbose=-1, n_jobs=4
)
XGB7_PARAMS = dict(
    n_estimators=8000, learning_rate=0.03, max_depth=2, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.5, reg_lambda=1.0, max_bin=256,
    tree_method="hist", device="cuda", eval_metric="auc", early_stopping_rounds=180
)

def boost_view7(view, idx, E):
    if view == "raw":
        return np.hstack([RAWV7[idx], CNT7[idx], KCNT7[idx]])
    elif view == "base":
        return np.hstack([RAWV7[idx], E[:, :NBASE7], KCNT7[idx, :NBASE7]])
    return np.hstack([RAWV7[idx], E, KCNT7[idx]])

def fit_glm_pair(Ea, Eb, Et, a, b, seed=0):
    if GEN7:
        Ga, Gb, Gt = gen_fold7(a, b, seed=seed)
        Ea_g = np.hstack([Ea, Ga])
        Eb_g = np.hstack([Eb, Gb]) if Eb is not None else None
        Et_g = np.hstack([Et, Gt])
    else:
        Ea_g, Eb_g, Et_g = Ea, Eb, Et
    sc = StandardScaler().fit(logit7(Ea_g))
    Za = sc.transform(logit7(Ea_g))
    Zt = sc.transform(logit7(Et_g))
    Zb = sc.transform(logit7(Eb_g)) if Eb_g is not None else None
    out = {}
    for gname, B in [("glm_gen", BASE7), ("glm_cnt", BASEC7)]:
        Xa = np.hstack([Za, B[a]])
        Xt = np.hstack([Zt, B[NTR7:]])
        m = LogisticRegression(C=0.3, max_iter=300, tol=1e-7, solver="newton-cholesky").fit(Xa, y7[a])
        za = m.decision_function(Xa)
        zt = m.decision_function(Xt)
        zb = m.decision_function(np.hstack([Zb, B[b]])) if Zb is not None else None
        out[gname] = (za, zb, zt)
    return out

def boost_fit7(kind, Xa, ya, za, n_trees=None, Xb=None, yb=None, zb=None, seed=42):
    if kind == "lgb":
        P = {**LGB7_PARAMS, "random_state": seed}
        if n_trees:
            P["n_estimators"] = n_trees
            m = lgb.LGBMClassifier(**P)
            return m.fit(Xa, ya, init_score=za, categorical_feature=CAT_IDX7), n_trees
        m = lgb.LGBMClassifier(**P)
        m.fit(Xa, ya, init_score=za, eval_set=[(Xb, yb)], eval_init_score=[zb], eval_metric="auc",
              categorical_feature=CAT_IDX7, callbacks=[lgb.early_stopping(150, verbose=False)])
        return m, m.best_iteration_
    else:
        P = {**XGB7_PARAMS, "random_state": seed}
        if n_trees:
            P = {k: v for k, v in P.items() if k != "early_stopping_rounds"}
            P["n_estimators"] = n_trees
            m = xgb.XGBClassifier(**P)
            return m.fit(Xa, ya, base_margin=za), n_trees
        m = xgb.XGBClassifier(**P)
        m.fit(Xa, ya, base_margin=za, eval_set=[(Xb, yb)], base_margin_eval_set=[zb], verbose=False)
        return m, m.best_iteration + 1

def boost_margin7(m, X, z, it=None):
    if isinstance(m, lgb.LGBMClassifier):
        return z + m.predict(X, raw_score=True)
    return m.predict(X, base_margin=z, output_margin=True, iteration_range=(0, it) if it else None)

# --- 4A. 10-Fold Pipeline: Encode once per fold, fit GLMs + GPU XGB boosters + CPU LGB booster ---
need_10f = [m for m in ["glm10_gen", "glm10_cnt", "gb10_xgb_s0", "gb10_cnt_xgb_s0", "gb10_lgb_s0", "xgb10_te20"] if m not in OOF7]
if need_10f:
    log7(f"Running 10-Fold GLM + Residual Boosters for: {need_10f}")
    oof_d = {m: np.zeros(NTR7) for m in need_10f}
    pred_d = {m: np.zeros(NTE7) for m in need_10f}
    its_d = {m: [] for m in need_10f if not m.startswith("glm")}
    test_idx = np.arange(NTR7, NTR7 + NTE7)

    for f, (a, b) in enumerate(FOLDS10):
        t_f = time.time()
        Ea, Eb, Et = te_fold7(a, b, smooth=20.0, inner=5, seed=0)
        glms = fit_glm_pair(Ea, Eb, Et, a, b, seed=0)

        if "glm10_gen" in oof_d:
            oof_d["glm10_gen"][b] = glms["glm_gen"][1]
            pred_d["glm10_gen"] += glms["glm_gen"][2] / 10.0
        if "glm10_cnt" in oof_d:
            oof_d["glm10_cnt"][b] = glms["glm_cnt"][1]
            pred_d["glm10_cnt"] += glms["glm_cnt"][2] / 10.0

        Xa_j = boost_view7("joint", a, Ea)
        Xb_j = boost_view7("joint", b, Eb)
        Xt_j = boost_view7("joint", test_idx, Et)

        if "gb10_xgb_s0" in oof_d:
            za, zb, zt = glms["glm_gen"]
            m, it = boost_fit7("xgb", Xa_j, y7[a], za, Xb=Xb_j, yb=y7[b], zb=zb, seed=42)
            its_d["gb10_xgb_s0"].append(it)
            oof_d["gb10_xgb_s0"][b] = boost_margin7(m, Xb_j, zb, it)
            pred_d["gb10_xgb_s0"] += boost_margin7(m, Xt_j, zt, it) / 10.0

        if "gb10_cnt_xgb_s0" in oof_d:
            za, zb, zt = glms["glm_cnt"]
            m, it = boost_fit7("xgb", Xa_j, y7[a], za, Xb=Xb_j, yb=y7[b], zb=zb, seed=137)
            its_d["gb10_cnt_xgb_s0"].append(it)
            oof_d["gb10_cnt_xgb_s0"][b] = boost_margin7(m, Xb_j, zb, it)
            pred_d["gb10_cnt_xgb_s0"] += boost_margin7(m, Xt_j, zt, it) / 10.0

        if "gb10_lgb_s0" in oof_d:
            za, zb, zt = glms["glm_gen"]
            m, it = boost_fit7("lgb", Xa_j, y7[a], za, Xb=Xb_j, yb=y7[b], zb=zb, seed=42)
            its_d["gb10_lgb_s0"].append(it)
            oof_d["gb10_lgb_s0"][b] = boost_margin7(m, Xb_j, zb, it)
            pred_d["gb10_lgb_s0"] += boost_margin7(m, Xt_j, zt, it) / 10.0

        if "xgb10_te20" in oof_d:
            Xa_b = boost_view7("base", a, Ea)
            Xb_b = boost_view7("base", b, Eb)
            Xt_b = boost_view7("base", test_idx, Et)
            P_plain = {**XGB7_PARAMS, "learning_rate": 0.03, "random_state": 42}
            m = xgb.XGBClassifier(**P_plain)
            m.fit(Xa_b, y7[a], eval_set=[(Xb_b, y7[b])], verbose=False)
            it = m.best_iteration + 1
            its_d["xgb10_te20"].append(it)
            oof_d["xgb10_te20"][b] = m.predict(Xb_b, output_margin=True, iteration_range=(0, it))
            pred_d["xgb10_te20"] += m.predict(Xt_b, output_margin=True, iteration_range=(0, it)) / 10.0

        msg = [f"10F fold {f} ({time.time()-t_f:.0f}s):"]
        for mname in need_10f:
            msg.append(f"{mname}={roc_auc_score(y7[b], oof_d[mname][b]):.5f}")
        log7(" | ".join(msg))
        gc.collect()

    for mname in need_10f:
        finish7(mname, oof_d[mname], pred_d[mname], pred_d[mname])
    gc.collect()

# --- 4B. 5-Fold Suite (Different Fold Geometry + Encoding Seed 1 for Diversity) ---
FOLDS5 = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR7), y7))
need_5f = [m for m in ["glm5_cnt", "gb5_xgb_s1", "gb5_lgb_s1", "gb5_raw_xgb"] if m not in OOF7]
if need_5f:
    log7(f"Running 5-Fold GLM + Residual Boosters (teseed=1) for: {need_5f}")
    oof5 = {m: np.zeros(NTR7) for m in need_5f}
    pred5 = {m: np.zeros(NTE7) for m in need_5f}
    its5 = {m: [] for m in need_5f if not m.startswith("glm")}
    test_idx = np.arange(NTR7, NTR7 + NTE7)

    for f, (a, b) in enumerate(FOLDS5):
        t_f = time.time()
        Ea, Eb, Et = te_fold7(a, b, smooth=20.0, inner=5, seed=1)
        glms = fit_glm_pair(Ea, Eb, Et, a, b, seed=1)

        if "glm5_cnt" in oof5:
            oof5["glm5_cnt"][b] = glms["glm_cnt"][1]
            pred5["glm5_cnt"] += glms["glm_cnt"][2] / 5.0

        Xa_j = boost_view7("joint", a, Ea)
        Xb_j = boost_view7("joint", b, Eb)
        Xt_j = boost_view7("joint", test_idx, Et)

        if "gb5_xgb_s1" in oof5:
            za, zb, zt = glms["glm_gen"]
            m, it = boost_fit7("xgb", Xa_j, y7[a], za, Xb=Xb_j, yb=y7[b], zb=zb, seed=137)
            its5["gb5_xgb_s1"].append(it)
            oof5["gb5_xgb_s1"][b] = boost_margin7(m, Xb_j, zb, it)
            pred5["gb5_xgb_s1"] += boost_margin7(m, Xt_j, zt, it) / 5.0

        if "gb5_lgb_s1" in oof5:
            za, zb, zt = glms["glm_gen"]
            m, it = boost_fit7("lgb", Xa_j, y7[a], za, Xb=Xb_j, yb=y7[b], zb=zb, seed=137)
            its5["gb5_lgb_s1"].append(it)
            oof5["gb5_lgb_s1"][b] = boost_margin7(m, Xb_j, zb, it)
            pred5["gb5_lgb_s1"] += boost_margin7(m, Xt_j, zt, it) / 5.0

        if "gb5_raw_xgb" in oof5:
            za, zb, zt = glms["glm_cnt"]
            Xa_r = boost_view7("raw", a, Ea)
            Xb_r = boost_view7("raw", b, Eb)
            Xt_r = boost_view7("raw", test_idx, Et)
            m, it = boost_fit7("xgb", Xa_r, y7[a], za, Xb=Xb_r, yb=y7[b], zb=zb, seed=42)
            its5["gb5_raw_xgb"].append(it)
            oof5["gb5_raw_xgb"][b] = boost_margin7(m, Xb_r, zb, it)
            pred5["gb5_raw_xgb"] += boost_margin7(m, Xt_r, zt, it) / 5.0

        msg = [f"5F fold {f} ({time.time()-t_f:.0f}s):"]
        for mname in need_5f:
            msg.append(f"{mname}={roc_auc_score(y7[b], oof5[mname][b]):.5f}")
        log7(" | ".join(msg))
        gc.collect()

    for mname in need_5f:
        finish7(mname, oof5[mname], pred5[mname], pred5[mname])
    gc.collect()

# ==============================================================================
# 5. Grand Out-of-Fold Rank Hill-Climbing & Logit Stack (Exp 7 + V6 Models)
# ==============================================================================
pool_oof = {}
pool_test = {}
for k in OOF7:
    pool_oof[k] = OOF7[k]
    pool_test[k] = PRED7[k]

# Add our best diverse cached V6 models (10-fold GBDTs, Spline GLM, PLR-ResNet, MLP)
v6_candidates = [
    ("V6_LGB_3L_Dig2_10F", "lgb_dig2_10f_oof", "lgb_dig2_10f_test"),
    ("V6_Spline_GLM3_10F", "glm_dig3_10f_oof", "glm_dig3_10f_test"),
    ("V6_LGB_ET_Dig3_10F", "lgb_et_10f_oof", "lgb_et_10f_test"),
    ("V6_LGB_CleanPairs", "lgb_clean_oof", "lgb_clean_test"),
    ("V6_LGB_3L_Dig3", "lgb_dig3_oof", "lgb_dig3_test"),
    ("V6_PLR_WideResNet", "nn_dig3_oof", "nn_dig3_test"),
    ("V6_PyTorch_MLP", "mlp_oof", "mlp_test"),
]
for label, oof_var, te_var in v6_candidates:
    if oof_var in globals() and np.std(globals()[oof_var]) > 0:
        pool_oof[label] = globals()[oof_var]
        pool_test[label] = globals()[te_var]

NAMES_ALL = list(pool_oof.keys())
print("\n" + "="*75)
print("SINGLE MODEL OOF ROC AUC LEADERBOARD (EXP 7 + CACHED V6)")
print("="*75)
single_aucs = {n: roc_auc_score(y7, pool_oof[n]) for n in NAMES_ALL}
for n in sorted(NAMES_ALL, key=lambda k: single_aucs[k], reverse=True):
    print(f"  {n:24s}: {single_aucs[n]:.6f}")

R_oof = np.column_stack([rankdata(pool_oof[n]) / NTR7 for n in NAMES_ALL]).astype(np.float32)
R_test = np.column_stack([rankdata(pool_test[n]) / NTE7 for n in NAMES_ALL]).astype(np.float32)

def fast_auc_hist(yy, s, nbins=25000):
    s_min, s_max = float(s.min()), float(s.max())
    if s_max <= s_min:
        return 0.5
    b = np.clip(((s - s_min) * ((nbins - 1) / (s_max - s_min))).astype(np.int32), 0, nbins - 1)
    pos = np.bincount(b, weights=yy, minlength=nbins)
    tot = np.bincount(b, minlength=nbins)
    neg = tot - pos
    cum_neg = np.cumsum(neg)
    auc_num = np.sum(pos * (cum_neg - 0.5 * neg))
    return float(auc_num / (pos.sum() * neg.sum()))

def hill_climb_auc(Rm, yy, iters=100):
    yy_f = yy.astype(np.float64)
    w = np.zeros(Rm.shape[1], dtype=np.float64)
    cur = np.zeros(len(yy), dtype=np.float32)
    best = 0.0
    for _ in range(iters):
        sc = [fast_auc_hist(yy_f, cur + Rm[:, j]) for j in range(Rm.shape[1])]
        j = int(np.argmax(sc))
        if sc[j] <= best + 1e-7:
            break
        best, cur, w[j] = sc[j], cur + Rm[:, j], w[j] + 1.0
    return w / np.maximum(w.sum(), 1.0)

def rlogit7(r):
    return logit7(np.clip(r, 1e-6, 1 - 1e-6))

# Nested 5-Fold Out-of-Fold Evaluation of Combiners (Hill Climbing vs Rank-Logit Stack vs Exp7 Rank Mean)
exp7_only_idx = [i for i, n in enumerate(NAMES_ALL) if not n.startswith("V6_")]
nested_eval = {
    "exp7_rank_mean": R_oof[:, exp7_only_idx].mean(1),
    "hill_all": np.zeros(NTR7),
    "stack_all": np.zeros(NTR7),
}
for a, b in FOLDS5:
    w_fold = hill_climb_auc(R_oof[a], y7[a], iters=80)
    nested_eval["hill_all"][b] = R_oof[b] @ w_fold
    st_fold = LogisticRegression(C=0.2, max_iter=1000).fit(rlogit7(R_oof[a]), y7[a])
    nested_eval["stack_all"][b] = st_fold.decision_function(rlogit7(R_oof[b]))

print("\n" + "="*75)
print("OUT-OF-FOLD COMBINER COMPARISON (NESTED CV)")
print("="*75)
for k, v in nested_eval.items():
    print(f"  {k:24s}: {roc_auc_score(y7, v):.6f}")

# Fit final Hill-Climbing weights on full OOF
w_hill = hill_climb_auc(R_oof, y7, iters=120)
oof_hill_full = R_oof @ w_hill
test_hill_full = R_test @ w_hill

print("\nFinal Hill-Climbing Weights:")
for n, w in zip(NAMES_ALL, w_hill):
    if w > 0:
        print(f"  {n:24s}: {w:.4f} (OOF {single_aucs[n]:.6f})")
final_exp7_auc = roc_auc_score(y7, oof_hill_full)
print(f"\n>>> FINAL EXP 7 HILL-CLIMB ENSEMBLE OOF AUC: {final_exp7_auc:.6f} <<<")
print(f"    (Gain vs V6 Meta-Stacker 0.950490: +{final_exp7_auc - 0.950490:.6f})")

sub_exp7 = pd.DataFrame({"id": te7["id"], TARGET: test_hill_full})
sub_exp7.to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({len(sub_exp7)} rows, NaN={sub_exp7[TARGET].isna().sum()}, range=[{sub_exp7[TARGET].min():.4f}, {sub_exp7[TARGET].max():.4f}])")


tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[  27.2s] Built KMAT7: 92 keys (84 base + 8 joint), RAWV7: 30 cols
[  57.9s] Loaded Exp 7 cache with members: ['glm10_gen', 'glm10_cnt', 'gb10_xgb_s0', 'gb10_cnt_xgb_s0', 'gb10_lgb_s0', 'xgb10_te20', 'glm5_cnt', 'gb5_xgb_s1', 'gb5_lgb_s1', 'gb5_raw_xgb']

SINGLE MODEL OOF ROC AUC LEADERBOARD (EXP 7 + CACHED V6)
  gb10_lgb_s0             : 0.951795
  gb10_xgb_s0             : 0.951743
  gb10_cnt_xgb_s0         : 0.951664
  gb5_lgb_s1              : 0.951494
  gb5_xgb_s1              : 0.951399
  glm10_cnt               : 0.951327
  gb5_raw_xgb             : 0.951253
  glm10_gen               : 0.951099
  glm5_cnt                : 0.951053
  xgb10_te20              : 0.950876
  V6_LGB_3L_Dig2_10F      : 0.950124
  V6_Spline_GLM3_10F      : 0.949736
  V6_LGB_ET_Dig3_10F      : 0.949661
  V6_LGB_CleanPairs       : 0.949591
  V6_LGB_3L_Dig3          : 0.949576
  V6_PLR_WideResNet       : 0.948658
  V6_PyTorch_MLP          : 0.946093

OUT-OF-FOLD COMBINER COMPARISON (NESTED CV)
  exp7_rank_m

In [17]:
# ==============================================================================
# Step 16: Exp 9 — Multi-Resolution/Multi-Smoothing GLM + 3-Leaf Residual Boosters
#          + Per-Value/Digit/Token Categorical Embedding RealMLP (pytabkit)
#          + 100%-Data Full Refits + Regularized C=0.01 Logit-Rank Stacker
# ==============================================================================
import os, sys, io, gc, time, logging, subprocess, importlib
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer, KBinsDiscretizer, TargetEncoder

T0_EXP9 = time.time()
def log9(*a):
    print(f"[{time.time() - T0_EXP9:6.1f}s]", *a, flush=True)

import shutil, glob
EXP9_CACHE_FILE = "preds_exp9.npz"
if not os.path.exists(EXP9_CACHE_FILE):
    _hits9 = sorted(glob.glob(f"/kaggle/input/**/{EXP9_CACHE_FILE}", recursive=True))
    if _hits9:
        shutil.copy2(_hits9[0], EXP9_CACHE_FILE)
        log9(f"Copied {EXP9_CACHE_FILE} from {_hits9[0]} into /kaggle/working/")
OOF9, FOLDAVG9, REFIT9, PRED9 = {}, {}, {}, {}
if os.path.exists(EXP9_CACHE_FILE):
    _c9 = np.load(EXP9_CACHE_FILE)
    for _k in _c9.files:
        if _k.startswith("oof_"):
            OOF9[_k[4:]] = _c9[_k]
        elif _k.startswith("foldavg_"):
            FOLDAVG9[_k[8:]] = _c9[_k]
        elif _k.startswith("refit_"):
            REFIT9[_k[6:]] = _c9[_k]
        elif _k.startswith("pred_"):
            PRED9[_k[5:]] = _c9[_k]
    log9(f"Loaded Exp 9 cache with members: {list(OOF9.keys())}")

def save_exp9_cache():
    payload = {}
    for k in OOF9:
        payload[f"oof_{k}"] = OOF9[k].astype(np.float32)
        payload[f"foldavg_{k}"] = FOLDAVG9[k].astype(np.float32)
        payload[f"refit_{k}"] = REFIT9[k].astype(np.float32)
        payload[f"pred_{k}"] = PRED9[k].astype(np.float32)
    np.savez_compressed(EXP9_CACHE_FILE, **payload)

def finish9(name, oof, foldavg, refit):
    OOF9[name], FOLDAVG9[name], REFIT9[name] = oof, foldavg, refit
    PRED9[name] = (rankdata(foldavg) + rankdata(refit)) / (2.0 * NTE7)
    sp = pd.Series(foldavg).corr(pd.Series(refit), method="spearman")
    log9(f"==> {name}: OOF AUC {roc_auc_score(y7, oof):.6f} | foldavg vs refit spearman {sp:.4f}")
    save_exp9_cache()

test_idx9 = np.arange(NTR7, NTR7 + NTE7)

# ==============================================================================
# PART A: 100%-Data Full Refits for Top Exp 7 Models (Fixes 90%-Data Rare-Value Noise)
# ==============================================================================
if np.allclose(FOLDAVG7["gb10_lgb_s0"], REFIT7["gb10_lgb_s0"]):
    log9("Part A: Computing true 100%-data full refit encodings & models for Exp 7 10F suite...")
    Ea_all7, _, Et_all7 = te_fold7(ALL7[0], ALL7[1], smooth=20.0, inner=5, seed=0)
    glms_all7 = fit_glm_pair(Ea_all7, None, Et_all7, ALL7[0], ALL7[1], seed=0)

    finish7("glm10_gen", OOF7["glm10_gen"], FOLDAVG7["glm10_gen"], glms_all7["glm_gen"][2])
    finish7("glm10_cnt", OOF7["glm10_cnt"], FOLDAVG7["glm10_cnt"], glms_all7["glm_cnt"][2])

    Xa_all_j7 = boost_view7("joint", ALL7[0], Ea_all7)
    Xt_all_j7 = boost_view7("joint", test_idx9, Et_all7)

    za_g7, _, zt_g7 = glms_all7["glm_gen"]
    za_c7, _, zt_c7 = glms_all7["glm_cnt"]

    m_lgb_all, _ = boost_fit7("lgb", Xa_all_j7, y7, za_g7, n_trees=1100, seed=42)
    finish7("gb10_lgb_s0", OOF7["gb10_lgb_s0"], FOLDAVG7["gb10_lgb_s0"], boost_margin7(m_lgb_all, Xt_all_j7, zt_g7))

    m_xgb_all, _ = boost_fit7("xgb", Xa_all_j7, y7, za_g7, n_trees=950, seed=42)
    finish7("gb10_xgb_s0", OOF7["gb10_xgb_s0"], FOLDAVG7["gb10_xgb_s0"], boost_margin7(m_xgb_all, Xt_all_j7, zt_g7))

    m_cx_all, _ = boost_fit7("xgb", Xa_all_j7, y7, za_c7, n_trees=950, seed=137)
    finish7("gb10_cnt_xgb_s0", OOF7["gb10_cnt_xgb_s0"], FOLDAVG7["gb10_cnt_xgb_s0"], boost_margin7(m_cx_all, Xt_all_j7, zt_c7))

    del Ea_all7, Et_all7, Xa_all_j7, Xt_all_j7, m_lgb_all, m_xgb_all, m_cx_all
    gc.collect()
else:
    log9("Part A: Exp 7 100%-data full refits already computed and cached.")

# ==============================================================================
# PART B: Multi-Resolution Modulo-Decimal + Multi-Smoothing GLM + 3-Leaf Residual Boosters
# ==============================================================================
# Build 24 new generator-fingerprint keys:
#  - 6 pure first decimal digits (d1_*: floor(x*10)%10)
#  - 6 pure second decimal digits (d2_*: round(x*100)%10)
#  - 6 ones-place integer digits (u1_*: floor(abs(x))%10)
#  - 6 fine 50-quantile bins (q50_*)
extra_k9 = {}
for c in FLOATS7:
    x = full7[c].values
    extra_k9[f"d1_{c}"] = (np.floor(x * 10 + 1e-6).astype(np.int64) % 10).astype(np.int32)
    extra_k9[f"d2_{c}"] = (np.round(x * 100).astype(np.int64) % 10).astype(np.int32)
    extra_k9[f"u1_{c}"] = (np.floor(np.abs(x)).astype(np.int64) % 10).astype(np.int32)
    extra_k9[f"q50_{c}"] = pd.qcut(x, 50, labels=False, duplicates="drop").astype(np.int32)

ECODES9 = [pd.factorize(pd.Series(extra_k9[k]))[0].astype(np.int32) for k in extra_k9]
KMAT9 = np.column_stack([KMAT7] + ECODES9)
NCAT9 = np.append(NCAT7, [ec.max() + 1 for ec in ECODES9])
KCNT9 = np.column_stack([KCNT7] + [np.bincount(ec)[ec].astype(np.float32) for ec in ECODES9])
LOGC9 = np.log1p(KCNT9)
ZLOGC9 = ((LOGC9 - LOGC9.mean(0)) / (LOGC9.std(0) + 1e-6)).astype(np.float32)

# Quantile-normalised continuous numerics + counts for the Multi-Resolution GLM
qt9 = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000, random_state=42)
QNUM9 = qt9.fit_transform(np.hstack([full7[NUMS7].values.astype(np.float32), CNT7])).astype(np.float32)
BASE_MR9 = np.hstack([BASE7, QNUM9, ZLOGC9]).astype(np.float32)

# High-cardinality key indices (24 exact per-value keys + 6 int + 6 cents + 6 token-0 = 42 keys)
# We encode these at BOTH smooth=5.0 (local sharp rate) and smooth=60.0 (shrunk prior rate)
# in addition to smooth=20.0 across all 116 keys in KMAT9!
HC_IDX9 = list(range(24)) + [i for i, n in enumerate(KNAMES7) if n.startswith(("int_", "cents_", "t1_"))]
log9(f"Built KMAT9: {KMAT9.shape[1]} keys ({KMAT7.shape[1]} Exp7 + {len(ECODES9)} modulo-digit/q50), HC_IDX9={len(HC_IDX9)} multi-smooth keys")

def te_mr_fold9(a, b, inner=5, seed=2):
    ya = y7[a].astype(np.float64)
    Ka, Kt = KMAT9[a], KMAT9[NTR7:]
    Kb = KMAT9[b] if len(b) else None
    n_k = KMAT9.shape[1]
    n_hc = len(HC_IDX9)
    n_tot = n_k + 2 * n_hc

    Ea = np.zeros((len(a), n_tot), np.float32)
    Eb = np.zeros((len(b), n_tot), np.float32) if len(b) else None
    Et = np.zeros((NTE7, n_tot), np.float32)

    inner_f = list(StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), ya))
    # 1) All 116 keys at smooth=20.0
    for j in range(n_k):
        for i, o in inner_f:
            Ea[o, j] = te_rate7(Ka[i, j], ya[i], NCAT9[j], ya[i].mean(), 20.0)[Ka[o, j]]
        r20 = te_rate7(Ka[:, j], ya, NCAT9[j], ya.mean(), 20.0)
        if len(b):
            Eb[:, j] = r20[Kb[:, j]]
        Et[:, j] = r20[Kt[:, j]]

    # 2) High-cardinality keys at smooth=5.0 and smooth=60.0
    for idx_pos, j in enumerate(HC_IDX9):
        c5 = n_k + idx_pos
        c60 = n_k + n_hc + idx_pos
        for i, o in inner_f:
            Ea[o, c5] = te_rate7(Ka[i, j], ya[i], NCAT9[j], ya[i].mean(), 5.0)[Ka[o, j]]
            Ea[o, c60] = te_rate7(Ka[i, j], ya[i], NCAT9[j], ya[i].mean(), 60.0)[Ka[o, j]]
        r5 = te_rate7(Ka[:, j], ya, NCAT9[j], ya.mean(), 5.0)
        r60 = te_rate7(Ka[:, j], ya, NCAT9[j], ya.mean(), 60.0)
        if len(b):
            Eb[:, c5] = r5[Kb[:, j]]
            Eb[:, c60] = r60[Kb[:, j]]
        Et[:, c5] = r5[Kt[:, j]]
        Et[:, c60] = r60[Kt[:, j]]
    return Ea, Eb, Et

def fit_glm_mr9(Ea, Eb, Et, a, b, seed=2):
    if GEN7:
        Ga, Gb, Gt = gen_fold7(a, b, seed=seed)
        Ea_g = np.hstack([Ea, Ga])
        Eb_g = np.hstack([Eb, Gb]) if Eb is not None else None
        Et_g = np.hstack([Et, Gt])
    else:
        Ea_g, Eb_g, Et_g = Ea, Eb, Et
    sc = StandardScaler().fit(logit7(Ea_g))
    Za = sc.transform(logit7(Ea_g))
    Zt = sc.transform(logit7(Et_g))
    Zb = sc.transform(logit7(Eb_g)) if Eb_g is not None else None

    Xa = np.hstack([Za, BASE_MR9[a]])
    Xt = np.hstack([Zt, BASE_MR9[NTR7:]])
    m = LogisticRegression(C=0.2, max_iter=300, tol=1e-7, solver="newton-cholesky").fit(Xa, y7[a])
    za = m.decision_function(Xa)
    zt = m.decision_function(Xt)
    zb = m.decision_function(np.hstack([Zb, BASE_MR9[b]])) if Zb is not None else None
    return za, zb, zt

LGB9_3L_PARAMS = dict(
    n_estimators=8000, learning_rate=0.04, num_leaves=3, min_child_samples=120,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.45, reg_lambda=2.0,
    max_bin=255, force_col_wise=True, verbose=-1, n_jobs=4
)
XGB9_3L_PARAMS = dict(
    n_estimators=8000, learning_rate=0.03, grow_policy="lossguide", max_leaves=3, max_depth=0,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.45, reg_lambda=2.0, max_bin=256,
    tree_method="hist", device="cuda", eval_metric="auc", early_stopping_rounds=180
)

need_mr9 = [m for m in ["glm10_mr", "gb10_lgb3_mr", "gb10_xgb3_mr"] if m not in OOF9]
if need_mr9:
    log9(f"Part B: Running 10-Fold Multi-Resolution/Multi-Smoothing GLM + 3-Leaf Residual Boosters: {need_mr9}")
    oof_mr = {m: np.zeros(NTR7) for m in need_mr9}
    pred_mr = {m: np.zeros(NTE7) for m in need_mr9}
    its_mr = {m: [] for m in need_mr9 if not m.startswith("glm")}
    n_k9 = KMAT9.shape[1]

    for f, (a, b) in enumerate(FOLDS10):
        t_f = time.time()
        Ea, Eb, Et = te_mr_fold9(a, b, inner=5, seed=2)
        za, zb, zt = fit_glm_mr9(Ea, Eb, Et, a, b, seed=2)

        if "glm10_mr" in oof_mr:
            oof_mr["glm10_mr"][b] = zb
            pred_mr["glm10_mr"] += zt / 10.0

        Xa_mr = np.hstack([RAWV7[a], Ea[:, :n_k9], KCNT9[a]])
        Xb_mr = np.hstack([RAWV7[b], Eb[:, :n_k9], KCNT9[b]])
        Xt_mr = np.hstack([RAWV7[test_idx9], Et[:, :n_k9], KCNT9[test_idx9]])

        if "gb10_lgb3_mr" in oof_mr:
            m_l3 = lgb.LGBMClassifier(**{**LGB9_3L_PARAMS, "random_state": 2026 + f})
            m_l3.fit(Xa_mr, y7[a], init_score=za, eval_set=[(Xb_mr, y7[b])], eval_init_score=[zb],
                     eval_metric="auc", categorical_feature=CAT_IDX7,
                     callbacks=[lgb.early_stopping(150, verbose=False)])
            it_l3 = m_l3.best_iteration_
            its_mr["gb10_lgb3_mr"].append(it_l3)
            oof_mr["gb10_lgb3_mr"][b] = zb + m_l3.predict(Xb_mr, raw_score=True)
            pred_mr["gb10_lgb3_mr"] += (zt + m_l3.predict(Xt_mr, raw_score=True)) / 10.0

        if "gb10_xgb3_mr" in oof_mr:
            m_x3 = xgb.XGBClassifier(**{**XGB9_3L_PARAMS, "random_state": 2026 + f})
            m_x3.fit(Xa_mr, y7[a], base_margin=za, eval_set=[(Xb_mr, y7[b])], base_margin_eval_set=[zb], verbose=False)
            it_x3 = m_x3.best_iteration + 1
            its_mr["gb10_xgb3_mr"].append(it_x3)
            oof_mr["gb10_xgb3_mr"][b] = m_x3.predict(Xb_mr, base_margin=zb, output_margin=True, iteration_range=(0, it_x3))
            pred_mr["gb10_xgb3_mr"] += m_x3.predict(Xt_mr, base_margin=zt, output_margin=True, iteration_range=(0, it_x3)) / 10.0

        msg = [f"10F-MR fold {f} ({time.time()-t_f:.0f}s):"]
        for mname in need_mr9:
            msg.append(f"{mname}={roc_auc_score(y7[b], oof_mr[mname][b]):.5f}")
        log9(" | ".join(msg))
        del Ea, Eb, Et, Xa_mr, Xb_mr, Xt_mr
        gc.collect()

    log9("Computing 100%-data full refit for Multi-Resolution 10F models...")
    Ea_all9, _, Et_all9 = te_mr_fold9(ALL7[0], ALL7[1], inner=5, seed=2)
    za_all9, _, zt_all9 = fit_glm_mr9(Ea_all9, None, Et_all9, ALL7[0], ALL7[1], seed=2)
    if "glm10_mr" in oof_mr:
        finish9("glm10_mr", oof_mr["glm10_mr"], pred_mr["glm10_mr"], zt_all9)

    Xa_all_mr = np.hstack([RAWV7[ALL7[0]], Ea_all9[:, :n_k9], KCNT9[ALL7[0]]])
    Xt_all_mr = np.hstack([RAWV7[test_idx9], Et_all9[:, :n_k9], KCNT9[test_idx9]])

    if "gb10_lgb3_mr" in oof_mr:
        n_tr = int(round(np.mean(its_mr["gb10_lgb3_mr"]) * 1.11))
        m_l3_all = lgb.LGBMClassifier(**{**LGB9_3L_PARAMS, "n_estimators": n_tr, "random_state": 2026})
        m_l3_all.fit(Xa_all_mr, y7, init_score=za_all9, categorical_feature=CAT_IDX7)
        finish9("gb10_lgb3_mr", oof_mr["gb10_lgb3_mr"], pred_mr["gb10_lgb3_mr"], zt_all9 + m_l3_all.predict(Xt_all_mr, raw_score=True))

    if "gb10_xgb3_mr" in oof_mr:
        n_tr = int(round(np.mean(its_mr["gb10_xgb3_mr"]) * 1.11))
        P_x3 = {k: v for k, v in XGB9_3L_PARAMS.items() if k != "early_stopping_rounds"}
        P_x3.update({"n_estimators": n_tr, "random_state": 2026})
        m_x3_all = xgb.XGBClassifier(**P_x3).fit(Xa_all_mr, y7, base_margin=za_all9)
        finish9("gb10_xgb3_mr", oof_mr["gb10_xgb3_mr"], pred_mr["gb10_xgb3_mr"], m_x3_all.predict(Xt_all_mr, base_margin=zt_all9, output_margin=True))

    del Ea_all9, Et_all9, Xa_all_mr, Xt_all_mr
    gc.collect()

# ==============================================================================
# PART C: From-Scratch RealMLP_TD_Classifier with Per-Value + Digit + GPT-2 Token Embeddings
# ==============================================================================
def ensure_pytabkit():
    importlib.invalidate_caches()
    try:
        import pytabkit  # noqa: F401
        return True
    except Exception:
        log9("Installing pytabkit==1.8.0...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytabkit==1.8.0"], capture_output=True)
        importlib.invalidate_caches()
        try:
            import pytabkit  # noqa: F401
            return True
        except Exception as e:
            log9(f"pytabkit install failed: {e}")
            return False

if "realmlp_tok5" not in OOF9 and ensure_pytabkit():
    import torch
    from pytabkit import RealMLP_TD_Classifier
    for _lname in ("lightning.pytorch", "pytorch_lightning", "lightning", "lightning_fabric"):
        logging.getLogger(_lname).setLevel(logging.ERROR)

    def build_realmlp_frames():
        m_tr = tr7[FEATS7].copy()
        m_te = te7[FEATS7].copy()
        m_cat = list(CATS7) + list(FLAGS7)
        for c in FLAGS7:
            m_tr[c] = m_tr[c].astype("int32")
            m_te[c] = m_te[c].astype("int32")
        m_num = list(NUMS7)
        dec4 = ["amount_usd", "hours_since_last_txn", "velocity_score", "distance_from_home_km"]

        def coded_pair(v_tr, v_te):
            codes, uniques = pd.factorize(v_tr)
            lookup = {u: i for i, u in enumerate(uniques)}
            return codes.astype("int32"), pd.Series(v_te).map(lookup).fillna(-1).astype("int32").values

        new_cat, combo_names = [], []
        for df in (m_tr, m_te):
            for c in dec4:
                df[f"_{c}_decimal"] = (df[c] % 1).round(2).astype("float32")

        # Integer parts + cents as categorical embeddings
        for c in m_num + [f"_{c}_decimal" for c in dec4]:
            name = f"{c}_cat_" if c in m_num else f"{c[1:]}_cat_"
            if c in m_num:
                m_tr[name], m_te[name] = coded_pair(np.floor(m_tr[c]), np.floor(m_te[c]))
            else:
                m_tr[name], m_te[name] = coded_pair(m_tr[c], m_te[c])
            new_cat.append(name)

        # First decimal digit (d1_*) + GPT-2 first subword token (t1_*) as categorical embeddings
        for c in FLOATS7:
            d1_name = f"d1_{c}_cat_"
            m_tr[d1_name], m_te[d1_name] = coded_pair(extra_k9[f"d1_{c}"][:NTR7], extra_k9[f"d1_{c}"][NTR7:])
            new_cat.append(d1_name)

        # Fine quantile bins
        for c, nb in [("merchant_risk_score", 7), ("hours_since_last_txn", 500), ("hours_since_last_txn", 1000)]:
            name = f"{c}_{nb}_qbin_"
            kb = KBinsDiscretizer(n_bins=nb, encode="ordinal", strategy="quantile", subsample=None)
            m_tr[name] = kb.fit_transform(m_tr[[c]]).ravel().astype("int32")
            m_te[name] = kb.transform(m_te[[c]]).ravel().astype("int32")
            new_cat.append(name)

        # Category interactions
        for cols in [("merchant_category", "card_type", "txn_count_last_24h"), ("merchant_category", "customer_age")]:
            name = "_".join(cols) + "_"
            j_tr = m_tr[cols[0]].astype(str)
            j_te = m_te[cols[0]].astype(str)
            for col in cols[1:]:
                j_tr = j_tr + "_" + m_tr[col].astype(str)
                j_te = j_te + "_" + m_te[col].astype(str)
            m_tr[name], m_te[name] = coded_pair(j_tr, j_te)
            new_cat.append(name)
            combo_names.append(name)

        for name in m_cat + new_cat:
            m_tr[name] = m_tr[name].astype("category")
            m_te[name] = m_te[name].astype("category")

        te_src = combo_names + ["hours_since_last_txn"]
        return m_tr, m_te, m_cat + new_cat, te_src

    m_tr9, m_te9, MLP_CATS9, TE_SRC9 = build_realmlp_frames()
    te_names9 = [f"_{c}TE" for c in TE_SRC9]
    log9(f"Part C: Training 5-Fold RealMLP_TD_Classifier ({m_tr9.shape[1] + len(te_names9)} cols, {len(MLP_CATS9)} categorical embeddings)...")

    REALMLP_PARAMS = dict(
        random_state=42, verbosity=0, val_metric_name="1-auc_ovr",
        n_ens=8, n_epochs=2, stop_epoch=2, batch_size=288, use_early_stopping=False,
        lr=0.053, wd=0.018, sq_mom=0.988, lr_sched="flat_anneal", wd_sched="cos_log_15",
        first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=16,
        hidden_sizes=[512, 256, 128], act="silu", p_drop=0.05, p_drop_sched="pow5",
        plr_hidden_1=16, plr_hidden_2=8, plr_act_name="gelu", plr_lr_factor=0.1151, plr_sigma=0.75,
        ls_eps=0.01, ls_eps_sched="sqrt_cos", add_front_scale=False, bias_init_mode="neg-uniform-dynamic-2",
        tfms=["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding", "l2_normalize"]
    )

    oof_rmlp = np.zeros(NTR7)
    pred_rmlp = np.zeros(NTE7)
    for f, (a, b) in enumerate(FOLDS5):
        t_f = time.time()
        X_tr_f, X_va_f, X_te_f = m_tr9.iloc[a].copy(), m_tr9.iloc[b].copy(), m_te9.copy()
        enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)
        X_tr_f[te_names9] = enc.fit_transform(X_tr_f[TE_SRC9], y7[a])
        X_va_f[te_names9] = enc.transform(X_va_f[TE_SRC9])
        X_te_f[te_names9] = enc.transform(X_te_f[TE_SRC9])

        rmodel = RealMLP_TD_Classifier(**REALMLP_PARAMS)
        with redirect_stdout(io.StringIO()):
            rmodel.fit(X_tr_f, y7[a], X_va_f, y7[b], cat_col_names=MLP_CATS9)
        oof_rmlp[b] = rmodel.predict_proba(X_va_f)[:, 1]
        pred_rmlp += rmodel.predict_proba(X_te_f)[:, 1] / 5.0
        log9(f"RealMLP fold {f} ({time.time()-t_f:.0f}s): AUC={roc_auc_score(y7[b], oof_rmlp[b]):.5f}")
        del X_tr_f, X_va_f, X_te_f, rmodel, enc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    finish9("realmlp_tok5", oof_rmlp, pred_rmlp, pred_rmlp)
    del m_tr9, m_te9
    gc.collect()

# ==============================================================================
# PART D: Honest Nested 5-Fold Combiner Comparison (Exp 9 + Exp 7 + Cached V6)
# ==============================================================================
pool9_oof = {}
pool9_test = {}
for k in OOF7:
    pool9_oof[k] = OOF7[k]
    pool9_test[k] = PRED7[k]
for k in OOF9:
    pool9_oof[k] = OOF9[k]
    pool9_test[k] = PRED9[k]
for label, oof_var, te_var in v6_candidates:
    if oof_var in globals() and np.std(globals()[oof_var]) > 0:
        pool9_oof[label] = globals()[oof_var]
        pool9_test[label] = globals()[te_var]

NAMES9_ALL = list(pool9_oof.keys())
single_aucs9 = {n: roc_auc_score(y7, pool9_oof[n]) for n in NAMES9_ALL}

print("\n" + "="*78)
print("EXP 9 + EXP 7 + V6 SINGLE MODEL OOF ROC AUC LEADERBOARD")
print("="*78)
for n in sorted(NAMES9_ALL, key=lambda k: single_aucs9[k], reverse=True):
    tag = " [NEW EXP 9]" if n in OOF9 else ""
    print(f"  {n:24s}: {single_aucs9[n]:.6f}{tag}")

R9_oof = np.column_stack([rankdata(pool9_oof[n]) / NTR7 for n in NAMES9_ALL]).astype(np.float32)
R9_test = np.column_stack([rankdata(pool9_test[n]) / NTE7 for n in NAMES9_ALL]).astype(np.float32)
L9_oof = rlogit7(R9_oof).astype(np.float64)
L9_test = rlogit7(R9_test).astype(np.float64)

# Report rank correlation of new models vs gb10_lgb_s0
ref_rank = R9_oof[:, NAMES9_ALL.index("gb10_lgb_s0")]
print("\nRank Correlation vs gb10_lgb_s0 (Lower = Higher Diversity):")
for n in OOF9:
    corr_v = np.corrcoef(ref_rank, R9_oof[:, NAMES9_ALL.index(n)])[0, 1]
    print(f"  {n:24s}: corr={corr_v:.4f} | OOF AUC={single_aucs9[n]:.6f}")

# Nested 5-Fold Evaluation of Combiners:
#  1) Exp 7 Baseline Hill-Climbing (without Exp 9 models)
#  2) Exp 9 Hill-Climbing (with Exp 9 models)
#  3) Exp 9 L2 Logit-Rank Stacker (C=0.01, newton-cholesky)
#  4) Exp 9 L2 Logit-Rank Stacker (C=0.003, newton-cholesky)
#  5) Blend (50% Hill-Climbing + 50% Logit-Rank Stacker C=0.01)
exp7_idx = [i for i, n in enumerate(NAMES9_ALL) if n not in OOF9]
nest9 = {
    "exp7_hill_baseline": np.zeros(NTR7),
    "exp9_hill_all": np.zeros(NTR7),
    "exp9_stack_c001": np.zeros(NTR7),
    "exp9_stack_c0003": np.zeros(NTR7),
    "exp9_blend_hs": np.zeros(NTR7),
}

for a, b in FOLDS5:
    w_e7 = hill_climb_auc(R9_oof[a][:, exp7_idx], y7[a], iters=100)
    nest9["exp7_hill_baseline"][b] = R9_oof[b][:, exp7_idx] @ w_e7

    w_e9 = hill_climb_auc(R9_oof[a], y7[a], iters=100)
    p_hc = R9_oof[b] @ w_e9
    nest9["exp9_hill_all"][b] = p_hc

    st1 = LogisticRegression(C=0.01, tol=1e-8, max_iter=300, solver="newton-cholesky").fit(L9_oof[a], y7[a])
    p_st1 = rankdata(st1.decision_function(L9_oof[b])) / len(b)
    nest9["exp9_stack_c001"][b] = p_st1

    st2 = LogisticRegression(C=0.003, tol=1e-8, max_iter=300, solver="newton-cholesky").fit(L9_oof[a], y7[a])
    p_st2 = rankdata(st2.decision_function(L9_oof[b])) / len(b)
    nest9["exp9_stack_c0003"][b] = p_st2

    nest9["exp9_blend_hs"][b] = 0.5 * (rankdata(p_hc) / len(b)) + 0.5 * p_st1

print("\n" + "="*78)
print("HONEST NESTED 5-FOLD COMBINER COMPARISON (EXP 9 vs EXP 7)")
print("="*78)
nest9_aucs = {k: roc_auc_score(y7, v) for k, v in nest9.items()}
for k, auc_v in sorted(nest9_aucs.items(), key=lambda t: t[1], reverse=True):
    diff = auc_v - nest9_aucs["exp7_hill_baseline"]
    print(f"  {k:24s}: {auc_v:.6f}  ({diff:+.6f} vs Exp 7 nested baseline)")

# Fit full-OOF combiners
w9_hill = hill_climb_auc(R9_oof, y7, iters=140)
oof9_hill = R9_oof @ w9_hill
test9_hill = R9_test @ w9_hill

st9_full = LogisticRegression(C=0.01, tol=1e-8, max_iter=300, solver="newton-cholesky").fit(L9_oof, y7)
oof9_stack = st9_full.decision_function(L9_oof)
test9_stack = st9_full.decision_function(L9_test)

oof9_blend = 0.5 * (rankdata(oof9_hill) / NTR7) + 0.5 * (rankdata(oof9_stack) / NTR7)
test9_blend = 0.5 * (rankdata(test9_hill) / NTE7) + 0.5 * (rankdata(test9_stack) / NTE7)

print("\nFull-OOF Hill-Climbing Weights (Exp 9 Pool):")
for n, w in zip(NAMES9_ALL, w9_hill):
    if w > 0:
        print(f"  {n:24s}: {w:.4f} (OOF {single_aucs9[n]:.6f})")

coef_s = pd.Series(st9_full.coef_.ravel(), index=NAMES9_ALL).sort_values(key=np.abs, ascending=False)
print("\nTop 10 Logit-Rank Stacker (C=0.01) Coefficients:")
for n, c_val in coef_s.head(10).items():
    print(f"  {n:24s}: {c_val:+.4f} (OOF {single_aucs9[n]:.6f})")

best_nest_name = max(["exp9_hill_all", "exp9_stack_c001", "exp9_blend_hs"], key=lambda k: nest9_aucs[k])
if best_nest_name == "exp9_hill_all":
    final_oof9, final_test9 = oof9_hill, test9_hill
elif best_nest_name == "exp9_stack_c001":
    final_oof9, final_test9 = rankdata(oof9_stack) / NTR7, rankdata(test9_stack) / NTE7
else:
    final_oof9, final_test9 = oof9_blend, test9_blend

print(f"\n>>> SELECTED COMBINER BY NESTED CV: {best_nest_name} (Nested OOF: {nest9_aucs[best_nest_name]:.6f} | Full OOF: {roc_auc_score(y7, final_oof9):.6f}) <<<")

sub_exp9 = pd.DataFrame({"id": te7["id"], TARGET: final_test9})
sub_exp9.to_csv("submission_exp9.csv", index=False)
sub_exp9.to_csv("submission.csv", index=False)
log9(f"Saved submission_exp9.csv and submission.csv ({len(sub_exp9)} rows, NaN={sub_exp9[TARGET].isna().sum()})")


[   0.2s] Loaded Exp 9 cache with members: ['glm10_mr', 'gb10_lgb3_mr', 'gb10_xgb3_mr', 'realmlp_tok5']
[   0.2s] Part A: Computing true 100%-data full refit encodings & models for Exp 7 10F suite...
[ 118.8s] ==> glm10_gen: OOF AUC 0.951099 | foldavg vs refit spearman 0.9997
[ 122.3s] ==> glm10_cnt: OOF AUC 0.951327 | foldavg vs refit spearman 0.9997
[ 216.1s] ==> gb10_lgb_s0: OOF AUC 0.951795 | foldavg vs refit spearman 0.9973
[ 238.3s] ==> gb10_xgb_s0: OOF AUC 0.951743 | foldavg vs refit spearman 0.9983
[ 260.2s] ==> gb10_cnt_xgb_s0: OOF AUC 0.951664 | foldavg vs refit spearman 0.9979
[ 189.1s] Built KMAT9: 116 keys (92 Exp7 + 24 modulo-digit/q50), HC_IDX9=42 multi-smooth keys

EXP 9 + EXP 7 + V6 SINGLE MODEL OOF ROC AUC LEADERBOARD
  gb10_lgb_s0             : 0.951795
  gb10_xgb_s0             : 0.951743
  gb10_cnt_xgb_s0         : 0.951664
  gb10_xgb3_mr            : 0.951499 [NEW EXP 9]
  gb5_lgb_s1              : 0.951494
  gb10_lgb3_mr            : 0.951472 [NEW EXP 9]
  gb5_xg

In [18]:
# ==============================================================================
# Step 17: Exp 10 — Target-First GPT-2 Language Models + Diverse Tabular Families
#          + Honest Multi-Split Nested CV Ablation & Saturation Study
# ==============================================================================
import os, json, time, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

T0_EXP10 = time.time()
def log10(*a):
    print(f"[{time.time() - T0_EXP10:6.1f}s]", *a, flush=True)

INPUT10 = Path("/kaggle/input")
WORK10 = Path("/kaggle/working")

def logit_rank10(p):
    """Map predictions onto normalized logit-rank scale (strictly monotone)."""
    r = (rankdata(p) - 0.5) / len(p)
    return logit(np.clip(r, 1e-6, 1.0 - 1e-6))

def fetch_kaggle_dataset(slug, dst_name, required_files):
    # First check if already mounted under /kaggle/input/**
    hits = []
    for rf in required_files:
        found = sorted(INPUT10.rglob(rf))
        # Prefer folder matching slug suffix
        slug_part = slug.split("/")[-1]
        matched = [p for p in found if slug_part in str(p)] or found
        if matched:
            hits.append(matched[0].parent)
    if hits and all((hits[0] / rf).exists() for rf in required_files):
        log10(f"Found {slug} mounted at {hits[0]}")
        return hits[0]

    dst = WORK10 / "ext_assets" / dst_name
    dst.mkdir(parents=True, exist_ok=True)
    if all((dst / rf).exists() for rf in required_files):
        log10(f"Using cached {dst}")
        return dst

    log10(f"Downloading {slug} via kaggle CLI...")
    res = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(dst), "--unzip"],
                         capture_output=True, text=True)
    if res.returncode != 0:
        log10(f"WARNING: failed to download {slug}: {res.stderr[-300:]}")
        return None
    missing = [rf for rf in required_files if not (dst / rf).exists()]
    if missing:
        log10(f"WARNING: {slug} missing files {missing}")
        return None
    log10(f"Downloaded {slug} OK -> {dst}")
    return dst

# Verify seed-42 5-fold scheme used by external Target-First LM predictions
fold42 = np.zeros(NTR7, dtype=np.int32)
for k, (_, va) in enumerate(FOLDS5):
    fold42[va] = k

ext_oof, ext_test = {}, {}

# 1) Masaya Kawamata GPT-2 Target-First LM (CC BY 4.0; Chris Deotte S6E10 recipe)
p_gpt2 = fetch_kaggle_dataset("masayakawamata/s6e11-gpt2-target-first-oof", "gpt2_masaya",
                              ["oof.npy", "test.npy", "fold_ids.npy"])
if p_gpt2 is not None:
    o = np.load(p_gpt2 / "oof.npy").astype(np.float64).ravel()
    t = np.load(p_gpt2 / "test.npy").astype(np.float64).ravel()
    fids = np.load(p_gpt2 / "fold_ids.npy").ravel()
    assert len(o) == NTR7 and len(t) == NTE7 and np.isfinite(o).all() and np.isfinite(t).all()
    assert np.array_equal(fids, fold42), "GPT-2 fold scheme mismatch!"
    sha = hashlib.sha256((p_gpt2 / "oof.npy").read_bytes()).hexdigest()[:16]
    ext_oof["LM_GPT2_masaya"], ext_test["LM_GPT2_masaya"] = o, t
    log10(f"LM_GPT2_masaya: OOF={roc_auc_score(y7, o):.6f} | fold42=OK | sha256={sha}")

# 2) Frozen Compact Target-First LMs (megayak LM + goodpjw2008 Seed 7)
p_lm7 = fetch_kaggle_dataset("goodpjw2008/s6e11-compact-lm-seed7", "compact_lm_seed7",
                             ["megalm_oof_sha181c.npy", "megalm_test_sha3168.npy", "lm7_oof_5fold.npy", "lm7_test_5fold.npy"])
if p_lm7 is not None:
    for name, fo, ft in [("LM_megayak", "megalm_oof_sha181c.npy", "megalm_test_sha3168.npy"),
                         ("LM_seed7", "lm7_oof_5fold.npy", "lm7_test_5fold.npy")]:
        o = np.load(p_lm7 / fo).astype(np.float64).ravel()
        t = np.load(p_lm7 / ft).astype(np.float64).ravel()
        assert len(o) == NTR7 and len(t) == NTE7 and np.isfinite(o).all() and np.isfinite(t).all()
        ext_oof[name], ext_test[name] = o, t
        log10(f"{name}: OOF={roc_auc_score(y7, o):.6f}")

# 3) Target-First GPT-2 Seed 2026 (goodpjw2008 / Matapanino replica)
p_gpt26 = fetch_kaggle_dataset("goodpjw2008/s6e11-target-first-gpt2-seed-2026", "gpt2_seed2026",
                               ["gpt2_s2026_oof_5fold.npy", "gpt2_s2026_test_5fold.npy"])
if p_gpt26 is not None:
    o = np.load(p_gpt26 / "gpt2_s2026_oof_5fold.npy").astype(np.float64).ravel()
    t = np.load(p_gpt26 / "gpt2_s2026_test_5fold.npy").astype(np.float64).ravel()
    assert len(o) == NTR7 and len(t) == NTE7 and np.isfinite(o).all() and np.isfinite(t).all()
    ext_oof["LM_GPT2_s2026"], ext_test["LM_GPT2_s2026"] = o, t
    log10(f"LM_GPT2_s2026: OOF={roc_auc_score(y7, o):.6f}")

# 4) Diverse Artifacts (EBM_dig, RealMLP_dig, RealMLP_te) + WideLM 201-member Benchmark Anchor
p_art = fetch_kaggle_dataset("goodpjw2008/s6e11-fraud-series-artifacts", "fraud_artifacts",
                             ["ebm_dig_oof.npy", "ebm_dig_test.npy",
                              "realmlp_dig_oof.npy", "realmlp_dig_test.npy",
                              "realmlp_te_oof.npy", "realmlp_te_test.npy",
                              "wide_lm_stack_nested.npy", "wide_lm_stack_test.npy"])
if p_art is not None:
    for name, fo, ft in [("ART_EBM_dig", "ebm_dig_oof.npy", "ebm_dig_test.npy"),
                         ("ART_RealMLP_dig", "realmlp_dig_oof.npy", "realmlp_dig_test.npy"),
                         ("ART_RealMLP_te", "realmlp_te_oof.npy", "realmlp_te_test.npy"),
                         ("BENCH_WideLM", "wide_lm_stack_nested.npy", "wide_lm_stack_test.npy")]:
        o = np.load(p_art / fo).astype(np.float64).ravel()
        t = np.load(p_art / ft).astype(np.float64).ravel()
        assert len(o) == NTR7 and len(t) == NTE7 and np.isfinite(o).all() and np.isfinite(t).all()
        ext_oof[name], ext_test[name] = o, t
        log10(f"{name}: OOF={roc_auc_score(y7, o):.6f}")

# Assemble full pool: Our Exp 7 + Exp 9 + V6 + Loaded Language Models / Artifacts
all10_oof = {}
all10_test = {}
for label, src_oof, src_test in [("OURS", pool9_oof, pool9_test), ("EXT", ext_oof, ext_test)]:
    for k in src_oof:
        all10_oof[k] = src_oof[k]
        all10_test[k] = src_test[k]

# Rank correlations of External LMs & Artifacts vs our gb10_lgb_s0
ref10 = rankdata(all10_oof["gb10_lgb_s0"]) / NTR7
print("\n" + "=" * 80)
print("NEW EXTERNAL MODELS VS OUR BEST SINGLE TREE (gb10_lgb_s0)")
print("=" * 80)
for k in ext_oof:
    r_k = rankdata(ext_oof[k]) / NTR7
    c = float(np.corrcoef(ref10, r_k)[0, 1])
    print(f"  {k:24s}: OOF={roc_auc_score(y7, ext_oof[k]):.6f} | rank_corr_vs_exp7={c:.4f}")

# Honest Multi-Split Nested Stacking Evaluation:
# Evaluate across 3 completely independent StratifiedKFold splits (seeds 7, 8, 9) + seed 42
def eval_stack_honest(member_names, C_val=0.01, splits=(7, 8, 9)):
    Z = np.column_stack([logit_rank10(all10_oof[m]) for m in member_names])
    ZT = np.column_stack([logit_rank10(all10_test[m]) for m in member_names])
    split_aucs = []
    for sp in splits:
        oof_sp = np.zeros(NTR7)
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=sp).split(Z, y7):
            m = LogisticRegression(C=C_val, tol=1e-8, max_iter=300, solver="newton-cholesky").fit(Z[tr], y7[tr])
            oof_sp[va] = m.decision_function(Z[va])
        split_aucs.append(roc_auc_score(y7, oof_sp))
    m_full = LogisticRegression(C=C_val, tol=1e-8, max_iter=300, solver="newton-cholesky").fit(Z, y7)
    test_pred = rankdata(m_full.decision_function(ZT)) / NTE7
    coefs = pd.Series(m_full.coef_.ravel(), index=member_names)
    return float(np.mean(split_aucs)), split_aucs, test_pred, coefs

ours_names = list(pool9_oof.keys())
exp7_only_names = [n for n in ours_names if n not in OOF9]
lm_names = [n for n in ["LM_GPT2_masaya", "LM_megayak", "LM_seed7", "LM_GPT2_s2026"] if n in all10_oof]
art_names = [n for n in ["ART_EBM_dig", "ART_RealMLP_dig", "ART_RealMLP_te"] if n in all10_oof]

configs = [
    ("1) Exp 7 Only (baseline)", exp7_only_names, 0.01),
    ("2) Exp 7 + Exp 9 (our models only)", ours_names, 0.01),
]
if "LM_GPT2_masaya" in all10_oof:
    configs.append(("3) Ours (Exp7+9) + 1 GPT-2 LM (masaya)", ours_names + ["LM_GPT2_masaya"], 0.01))
if lm_names:
    configs.append(("4) Ours (Exp7+9) + All 4 Target-First LMs", ours_names + lm_names, 0.01))
if lm_names and art_names:
    configs.append(("5) Ours (Exp7+9) + 4 LMs + EBM + RealMLP", ours_names + lm_names + art_names, 0.01))
if "BENCH_WideLM" in all10_oof:
    configs.append(("6a) BENCHMARK: WideLM solo", ["BENCH_WideLM"], 0.01))
    configs.append(("6b) BENCHMARK: WideLM + Ours (Exp7+9)", ["BENCH_WideLM"] + ours_names, 0.01))
    configs.append(("6c) BENCHMARK: WideLM + Ours + 4 LMs + Art", ["BENCH_WideLM"] + ours_names + lm_names + art_names, 0.01))

print("\n" + "=" * 80)
print("HONEST MULTI-SPLIT NESTED CV ABLATION (SPLITS = 7, 8, 9 | C = 0.01)")
print("=" * 80)
ablation_results = {}
for label, mlist, cval in configs:
    if len(mlist) == 1:
        solo_auc = roc_auc_score(y7, all10_oof[mlist[0]])
        print(f"  {label:44s}: mean={solo_auc:.6f} (solo)")
        ablation_results[label] = (solo_auc, [solo_auc]*3, rankdata(all10_test[mlist[0]]) / NTE7, pd.Series([1.0], index=mlist))
    else:
        mean_auc, sp_aucs, t_pred, coefs = eval_stack_honest(mlist, C_val=cval)
        ablation_results[label] = (mean_auc, sp_aucs, t_pred, coefs)
        print(f"  {label:44s}: mean={mean_auc:.6f} | splits={[round(x, 6) for x in sp_aucs]}")

# Also sweep regularization C on Stage 4 and Stage 5 to make sure C=0.01 isn't under/over-regularized
for target_label in ["4) Ours (Exp7+9) + All 4 Target-First LMs", "5) Ours (Exp7+9) + 4 LMs + EBM + RealMLP"]:
    if target_label in ablation_results:
        mlist = [c[1] for c in configs if c[0] == target_label][0]
        print(f"\nC-Sweep for [{target_label}]:")
        for cv in [0.002, 0.005, 0.01, 0.02, 0.05]:
            m_auc, _, _, _ = eval_stack_honest(mlist, C_val=cv)
            print(f"    C={cv:<6g} -> Nested Mean AUC = {m_auc:.6f}")

for key_lbl, slug in [
    ("3) Ours (Exp7+9) + 1 GPT-2 LM (masaya)", "submission_exp10_ours_1lm.csv"),
    ("4) Ours (Exp7+9) + All 4 Target-First LMs", "submission_exp10_ours_4lms.csv"),
    ("5) Ours (Exp7+9) + 4 LMs + EBM + RealMLP", "submission_exp10_ours_4lms_art.csv"),
    ("6b) BENCHMARK: WideLM + Ours (Exp7+9)", "submission_exp10_widelm_plus_ours.csv"),
]:
    if key_lbl in ablation_results:
        mean_auc, _, t_pred, coefs = ablation_results[key_lbl]
        pd.DataFrame({"id": te7["id"], TARGET: t_pred}).to_csv(slug, index=False)
        print(f"\nSaved {slug} (Nested AUC={mean_auc:.6f}) | Top weights:")
        print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(4).to_string())

log10("Exp 10 complete!")


[   0.8s] Found masayakawamata/s6e11-gpt2-target-first-oof mounted at /kaggle/input/datasets/masayakawamata/s6e11-gpt2-target-first-oof
[   1.2s] LM_GPT2_masaya: OOF=0.950027 | fold42=OK | sha256=3468aaaeb7851739
[   1.7s] Found goodpjw2008/s6e11-compact-lm-seed7 mounted at /kaggle/input/datasets/goodpjw2008/s6e11-compact-lm-seed7
[   2.2s] LM_megayak: OOF=0.950428
[   2.4s] LM_seed7: OOF=0.949448
[   2.7s] Found goodpjw2008/s6e11-target-first-gpt2-seed-2026 mounted at /kaggle/input/datasets/goodpjw2008/s6e11-target-first-gpt2-seed-2026
[   3.2s] LM_GPT2_s2026: OOF=0.950140
[   4.3s] Found goodpjw2008/s6e11-fraud-series-artifacts mounted at /kaggle/input/datasets/goodpjw2008/s6e11-fraud-series-artifacts
[   4.6s] ART_EBM_dig: OOF=0.949808
[   4.8s] ART_RealMLP_dig: OOF=0.948529
[   5.1s] ART_RealMLP_te: OOF=0.947710
[   5.3s] BENCH_WideLM: OOF=0.952565

NEW EXTERNAL MODELS VS OUR BEST SINGLE TREE (gb10_lgb_s0)
  LM_GPT2_masaya          : OOF=0.950027 | rank_corr_vs_exp7=0.9594
  LM_meg